# External-shot TS profile workflow

This notebook applies the repository workflow to an AUG shot outside the interval database. ELM treatment follows the legacy inter-ELM cycle selection: retain the 75% to 90% portion of every gap between consecutive ELMs.

In [ ]:
import sys
import importlib
from pathlib import Path
import numpy as np
import pandas as pd

project_root = Path.cwd().resolve().parent
sys.path.insert(0, str(project_root / 'src'))
sys.path.insert(0, str(project_root.parent / 'aug_separatrix'))

from ipfnpytools.plot import use_base_style
from ts_profile_fitter.loading import (load_elm_times, load_physics_inputs, load_ts_data)
import ts_profile_fitter.preprocess as preprocess_module
import ts_profile_fitter.plotting as plotting_module
import ts_profile_fitter.fitting as fitting_module
import ts_profile_fitter.loading as loading_module
importlib.reload(loading_module)
importlib.reload(preprocess_module)
importlib.reload(plotting_module)
importlib.reload(fitting_module)
from ts_profile_fitter.preprocess import (
    augment_measurement_errors, filter_elm_cycles, map_to_r_midplane,
    preprocess_ts_data,
)
from ts_profile_fitter.fitting import compute_channel_knots, fit_mbsplt_ne_Te_TS, fit_modified_tanh_profile, modified_tanh
from ts_profile_fitter.derived_quantities import compute_equilibrium_separatrix, compute_physics_separatrix
from ts_profile_fitter.plotting import plot_profile_comparison, plot_profile_summary, plot_profile_fit
from ts_profile_fitter.connection_length import get_connection_length
from ts_profile_fitter.figure_naming import make_fig_name

## Configuration

In [ ]:
shot_id = 32959
interval = {'t_begin': 4.1, 't_end': 4.5, 'label': 'external-shot test'}
equilibrium_name = 'EQH'
diagnostic = 'VTA'
mapping_method = 'ipfnpytools'
r_edge_offset = 0.008
use_elm_filter = True
elm_cycle_start_fraction = 0.75
elm_cycle_end_fraction = 0.95
output_directory = project_root / 'plots'
save_figures = True
show_figures = True

rhop_filter = {
    'core': {
        'rhop': 'rho_pol',
        'rhop_min': 0.2,
        'rhop_max': 0.97,
    },
    'edge': {
        'rhop': 'rho_pol',
        'rhop_min': 0.7,
        'rhop_max': 1.09,
    },
}

outlier_config = {
    'Te_edge': {
        'value_min': 0.0,
        'value_max': 1000.0,
        'error_max': 180,
        'rel_error_max': 0.5,
        'remove_zeros': True,
    },
    'Ne_edge': {
        'value_min': 0.0,
        #'value_max': 11e19,
        #'error_max': 0.55e19,
        'remove_zeros': True,
    },
    'Te_core': {
        'value_min': 0.0,
        #'value_max': 7000.0,
        #'error_max': None,
        'remove_zeros': True,
    },
    'Ne_core': {
        'value_min': 0.0,
        #'value_max': 1.3e20,
        #'error_max': None,
        'remove_zeros': True,
    },
}
mtanh_config = {'Te': {'deg_tail':1 , 'deg_top': 3,
                        'fix_a_tail': 0, 'fix_w' : None, 'fix_x0': None, 'y_scale': 1e3},
                 'Ne': {'deg_tail': 1, 'deg_top': 3, 'y_scale': 1e19}}

cold_dense_config = {
    'enabled': False,
    'te_max': 1.1,
    'ne_min': 1.0e19,
}
te_error_config = {
    'enabled': False,
    'fixed_error': 30.0,  # eV
    'relative_error': 0,  # fraction, not percent
    'source_column': 'Te_err',
    'output_column': 'Te_err_augmented',
    'fit_error_column': 'Te_err_augmented',
}
fixed_error_sweep = {
    'fixed_0eV': {'fixed_error': 0.0, 'relative_error': 0.0},
    'fixed_5eV': {'fixed_error': 5.0, 'relative_error': 0.0},
    'fixed_10eV': {'fixed_error': 10.0, 'relative_error': 0.0},
    'fixed_20eV': {'fixed_error': 20.0, 'relative_error': 0.0},
    'fixed_50eV': {'fixed_error': 50.0, 'relative_error': 0.0},
    'fixed_100eV': {'fixed_error': 100.0, 'relative_error': 0.0},
}
relative_error_sweep = {
    'relative_0pct': {'fixed_error': 0.0, 'relative_error': 0.00},
    'relative_2pct': {'fixed_error': 0.0, 'relative_error': 0.02},
    'relative_5pct': {'fixed_error': 0.0, 'relative_error': 0.05},
    'relative_10pct': {'fixed_error': 0.0, 'relative_error': 0.10},
    'relative_20pct': {'fixed_error': 0.0, 'relative_error': 0.20},
}
combined_error_sweep = {
    'fixed_5eV_rel_2pct': {'fixed_error': 5.0, 'relative_error': 0.02},
    'fixed_10eV_rel_5pct': {'fixed_error': 10.0, 'relative_error': 0.05},
    'fixed_20eV_rel_5pct': {'fixed_error': 20.0, 'relative_error': 0.05},
    'fixed_50eV_rel_10pct': {'fixed_error': 50.0, 'relative_error': 0.10},
}
spline_config = {
    'method': 'ondrej',
    'knot_strategy': 'balanced_quantile',
    'knot_count_shift': 0,
    'dR_fine': 0.002,
    'sigma_conv': 0.0,
    'dR_channel_max': 0.003,
    'ne_scale': 1.0e+19,
    'Te_scale': 1.0e+3,
    'max_step_scale': 3,
    'use_errors': False,
    's_factor': 1.0,
    'mono': 'decreasing',
    'maxiter': 400,
}

In [ ]:
# Alternative preprocessing configuration from configs/pipeline_config.yaml.
# Set use_good_yaml_config=False to use the original configuration above.
use_good_yaml_config = False

from ts_profile_fitter.preprocess import (
    apply_cold_dense_edge_filters,
    apply_distance_outlier_filters,
)


good_yaml_rhop_filter = {
    'core': {
        'rhop': 'rho_pol',
        'rhop_min': 0.2,
        'rhop_max': 0.97,
    },
}
good_yaml_outlier_config = {
    'Te_edge': {
        'value_min': 0.0,
        'value_max': 1000.0,
        'error_max': 180.0,
        'rel_error_max': None,
        'remove_zeros': True,
    },
    'Ne_edge': {
        'value_min': 0.0,
        'value_max': 1.1e20,
        'error_max': 5.5e18,
        'remove_zeros': True,
    },
    'Te_core': {
        'value_min': 0.0,
        'value_max': 7000.0,
        'error_max': None,
        'remove_zeros': True,
    },
    'Ne_core': {
        'value_min': 0.0,
        'value_max': 1.3e20,
        'error_max': None,
        'remove_zeros': True,
    },
}
good_yaml_cold_dense_config = {
    'enabled': True,
    'te_max': 1.1,
    'ne_min': 1.0e19,
}
good_yaml_distance_outlier_config = {
    'Ne_edge': {
        'enabled': False,
        'n_neighbors': 5,
        'min_points': 3,
        'min_value': 2.0e19,
        'max_relative_error': 0.05,
        'zscore': 5.0,
        'min_scale': 1.0e18,
        'short_group_max_points': 5,
        'te_floor': 1.1,
        'te_max_relative_error': 0.05,
    },
}

good_mtanh_config = {'Te': {'deg_tail': 1, 'deg_top': 3, 'fix_a_tail': None, 'fix_w': None, 'fix_x0': None, 'y_scale': 1e3},
                      'Ne': {'deg_tail': 2, 'deg_top': 3, 'y_scale': 1e19}}
    
good_yaml_spline_config = {
    'method': 'ipfn_spline',
    'knot_strategy': 'edge_biased_quantile',
    'knot_count_shift': -1,
    'dR_fine': 0.002,
    'ne_scale': 1.0e+19,
    'Te_scale': 1.0e+3,
    'max_step_scale': 3,
    'use_errors': True,
    's_factor': 1.0,
    'mono': 'decreasing',
    'maxiter': 400,
}
if use_good_yaml_config:
    active_rhop_filter = good_yaml_rhop_filter
    active_outlier_config = good_yaml_outlier_config
    active_cold_dense_config = good_yaml_cold_dense_config
    active_distance_outlier_config = good_yaml_distance_outlier_config
    active_mtanh_config = good_mtanh_config
    active_spline_config = good_yaml_spline_config
else:
    active_rhop_filter = rhop_filter
    active_outlier_config = outlier_config
    active_cold_dense_config = cold_dense_config
    active_distance_outlier_config = None
    active_mtanh_config = mtanh_config
    active_spline_config = spline_config

print('Active preprocessing configuration:', 'good YAML' if use_good_yaml_config else 'original cell 4')

## Load the external shot and ELM times

In [ ]:
ts_data = load_ts_data(shot_id, diagnostic=diagnostic, equilibrium=equilibrium_name)
elm_times_begin, elm_times_end = load_elm_times(shot_id)
overlap = (elm_times_end >= interval['t_begin']) & (elm_times_begin <= interval['t_end'])
elm_times_begin = elm_times_begin[overlap]
elm_times_end = elm_times_end[overlap]
print(f'Raw TS rows: {len(ts_data["data"])}')
print(f'ELMs overlapping interval: {len(elm_times_begin)}')

In [ ]:
ts_data['data']['z']

In [ ]:
for name in [
    'R_edge',
    'Z_edge',
    'Te_e',
    'SigTe_e',
    'Ne_e',
    'SigNe_e',
]:
    values = ts_data['vta'].getobject(name)
    values = np.asarray(values.data if hasattr(values, 'data') else values)
    print(name, values.shape)

## Existing preprocessing plus ELM cycle selection

In [ ]:
mapped, preprocess_metadata = preprocess_ts_data(
    ts_data, interval=interval, r_edge_offset=r_edge_offset,
    mapping_method=mapping_method, rhop_filter=active_rhop_filter,
    outlier_config=active_outlier_config, return_metadata=True, report_outliers=True
)
mapped, cold_dense_metadata = apply_cold_dense_edge_filters(
    mapped, active_cold_dense_config, report=True, return_metadata=True
)
mapped, distance_outlier_metadata = apply_distance_outlier_filters(
    mapped, active_distance_outlier_config, report=True, return_metadata=True
)
extra_removed = [
    metadata['removed_rows']
    for metadata in (cold_dense_metadata, distance_outlier_metadata)
    if not metadata['removed_rows'].empty
]
if extra_removed:
    preprocess_metadata['removed_rows'] = pd.concat(
        [preprocess_metadata['removed_rows'], *extra_removed],
        ignore_index=True,
    )
# Keep the existing ELM selection unchanged.
if use_elm_filter:
    mapped, elm_metadata = filter_elm_cycles(
        mapped, elm_times_begin, elm_times_end,
        cycle_start_fraction=elm_cycle_start_fraction,
        cycle_end_fraction=elm_cycle_end_fraction,
        return_metadata=True,
    )
else:
    elm_metadata = {'rows_before': len(mapped), 'rows_after': len(mapped), 'rows_removed': 0, 'cycle_windows': []}
mapped = map_to_r_midplane(mapped)
if te_error_config['enabled']:
    mapped = augment_measurement_errors(
        mapped,
        quantity='Te',
        error_column=te_error_config['source_column'],
        fixed_error=te_error_config['fixed_error'],
        relative_error=te_error_config['relative_error'],
        output_column=te_error_config['output_column'],
    )
else:
    mapped = mapped.copy()
    mapped[te_error_config['output_column']] = mapped[te_error_config['source_column']]
print(f'Elm Filtering:{elm_metadata}')
print(f"Initial number of channels: {preprocess_metadata['channels_before']}")
print(f"Channels removed: {preprocess_metadata['channels_removed']}")
print(f"Te fit error column: {te_error_config['fit_error_column']}")
print(mapped.groupby('system').size())

In [ ]:
removed_edge_ne = preprocess_metadata['removed_rows'].loc[
    lambda df: (
        (df['system'] == 'edge')
        & df['filter_reason'].str.contains('Ne_outlier', na=False)
    ),
    [
        'channel_index',
        'time',
        'r_midplane',
        'rho_pol',
        'Te',
        'Te_err',
        'Ne',
        'Ne_err',
        'filter_reason',
    ],
].sort_values('r_midplane')

print(removed_edge_ne.to_string(index=False))

## Fit Te and Ne

In [ ]:
x_te = mapped['r_midplane'].to_numpy(float)
y_te = mapped['Te'].to_numpy(float)
yerr_te = mapped[te_error_config['fit_error_column']].to_numpy(float)
x_ne = mapped['r_midplane'].to_numpy(float)
y_ne = mapped['Ne'].to_numpy(float)
yerr_ne = mapped['Ne_err'].to_numpy(float)

mtanh_te = fit_modified_tanh_profile(
    x_te, y_te, yerr_te,
    deg_tail=active_mtanh_config['Te']['deg_tail'],
    deg_top=active_mtanh_config['Te']['deg_top'],
    fix_a_tail=active_mtanh_config['Te']['fix_a_tail'],
    fix_w=active_mtanh_config['Te']['fix_w'],
    fix_x0=active_mtanh_config['Te']['fix_x0'],
    y_scale=active_mtanh_config['Te']['y_scale'],
)
mtanh_ne = fit_modified_tanh_profile(
    x_ne, y_ne, yerr_ne,
    deg_tail=active_mtanh_config['Ne']['deg_tail'],
    deg_top=active_mtanh_config['Ne']['deg_top'],
    y_scale=active_mtanh_config['Ne']['y_scale'],
)

spline_input = (
    mapped.rename(columns={'r_midplane': 'R_mid', 'Ne': 'ne', 'Ne_err': 'ne_err'})
    .dropna(subset=['R_mid', 'ne', 'ne_err', 'Te', 'Te_err'])
)
knots = compute_channel_knots(
    spline_input,
    preprocess_metadata['channels_before'],
    preprocess_metadata['channels_removed'],
    knot_count_shift=active_spline_config['knot_count_shift'],
    strategy=active_spline_config['knot_strategy'],
)
knots = np.asarray(knots, dtype=float)
print(
    f"Spline strategy: {active_spline_config['knot_strategy']}; "
    f"knot shift: {active_spline_config['knot_count_shift']:+d}; "
    f"input rows: {len(spline_input)}; knots: {len(knots)}"
)

spline_method = active_spline_config['method'].lower()
if spline_method in {'ondrej', 'mbsplt', 'monotonic_bspline'}:
    ds_out, knots, p_ne, pcov_ne, p_te, pcov_te = fit_mbsplt_ne_Te_TS(
        spline_input,
        dR_fine=active_spline_config['dR_fine'],
        sigma_conv=active_spline_config['sigma_conv'],
        R_knots=knots,
        dR_channel_max=active_spline_config['dR_channel_max'],
        max_step_scale=active_spline_config['max_step_scale'],
        use_errors=active_spline_config['use_errors'],
        ne_scale=active_spline_config['ne_scale'],
        Te_scale=active_spline_config['Te_scale'],
    )
elif spline_method == 'ipfn_spline':
    ds_out, knots, p_ne, pcov_ne, p_te, pcov_te = fitting_module.fit_ipfn_spline_ne_Te_TS(
        spline_input,
        dR_fine=active_spline_config['dR_fine'],
        R_knots=knots,
        ne_scale=active_spline_config['ne_scale'],
        Te_scale=active_spline_config['Te_scale'],
        s_factor=active_spline_config['s_factor'],
        mono=active_spline_config['mono'],
        maxiter=active_spline_config['maxiter'],
    )
else:
    raise ValueError(
        f"Unknown spline method {active_spline_config['method']!r}; "
        "choose 'ondrej' or 'ipfn_spline'."
    )

print(f'Spline method: {spline_method}')
print(f'Fit points: {len(spline_input)}; spline points: {len(ds_out)}')
print(f'Modified-tanh Te parameters: {mtanh_te[0]}')
print(f'Modified-tanh Ne parameters: {mtanh_ne[0]}')

## Compute equilibrium and physics separatrices

In [ ]:
eqh_result = compute_equilibrium_separatrix(ts_data['equ'], interval, rho_sep=1.0)

# The legacy connection-length provider is used only for the optional physics model.
from get_connection_length import get_LcSOL
physics_inputs = load_physics_inputs(shot_id, interval, equilibrium=equilibrium_name, dWdt_method = 'paper')
physics_inputs['geometry']['L_par'] = get_connection_length(shot_id, interval, provider=get_LcSOL)

R_spline = ds_out['R_mid_spl'].to_numpy(float)
spline_result = compute_physics_separatrix(
    R_spline, ds_out['Te_spl'].to_numpy(float), ds_out['ne_spl'].to_numpy(float),
    power_balance=physics_inputs['power_balance'], geometry=physics_inputs['geometry'],
    x_grid=R_spline, fit_method='spline',
)
R_mtanh = np.linspace(min(x_te.min(), x_ne.min()), max(x_te.max(), x_ne.max()), 500)
mtanh_result = compute_physics_separatrix(
    R_mtanh,
    __import__('ts_profile_fitter.fitting', fromlist=['modified_tanh']).modified_tanh(R_mtanh, *mtanh_te[0]),
    __import__('ts_profile_fitter.fitting', fromlist=['modified_tanh']).modified_tanh(R_mtanh, *mtanh_ne[0]),
    power_balance=physics_inputs['power_balance'], geometry=physics_inputs['geometry'],
    x_grid=R_mtanh, fit_method='mtanh',
)
print({
    'Rsep_eqh': eqh_result['Rsep_eqh'],
    'Rsep_spline': spline_result['Rsep'],
    'Rsep_mtanh': mtanh_result['Rsep'],
})

## Profile comparison plots

In [ ]:
import matplotlib.pyplot as plt


use_base_style()
core = mapped[mapped['system'] == 'core']
edge = mapped[mapped['system'] == 'edge']
for quantity, mtanh_result_fit, y_label in [('Te', mtanh_te, 'Te [eV]'), ('Ne', mtanh_ne, 'Ne [m$^{-3}$]')]:
    fig, ax = plot_profile_comparison(
        core['r_midplane'], edge['r_midplane'], core[quantity], edge[quantity],
        core[f'{quantity}_err'], edge[f'{quantity}_err'], ds_out, quantity,
        mtanh_popt=mtanh_result_fit[0], knot_pos=knots,
        time_c=core['time'], time_e=edge['time'],
        Rsep_eqh=eqh_result['Rsep_eqh'], Rsep_spline=spline_result['Rsep'],
        Rsep_mtanh=mtanh_result['Rsep'], nshot=shot_id, interval=interval,
        directory=str(output_directory), fig_name=make_fig_name(shot_id, equilibrium_name, len(knots), interval, f'{quantity.lower()}_elm_cycle'),
        y_labels=y_label, save_fig=save_figures, show=False, close=False,
        y_maxs=2000, x_mins= 2.10, x_maxs = 2.22, y_mins=-300
    )
ax.legend(loc='upper left')
plt.show()

## Summary plot

In [ ]:
fig, ax= plot_profile_summary(
    core['r_midplane'], edge['r_midplane'],
    core['Ne'], edge['Ne'], core['Te'], edge['Te'],
    ds_out, mtanh_te[0], mtanh_ne[0],
    lambda_spline=spline_result['lam_Te_grid'],
    lambda_mtanh=mtanh_result['lam_Te_grid'],
    q_sol_spline=spline_result['q_par_P_SOL_grid'],
    q_cond_spline=spline_result['q_par_SH_grid'],
    q_sol_mtanh=mtanh_result['q_par_P_SOL_grid'],
    q_cond_mtanh=mtanh_result['q_par_SH_grid'],
    R_spline=spline_result['R_grid'], R_mtanh=mtanh_result['R_grid'],
    ne_err_core=core['Ne_err'], ne_err_edge=edge['Ne_err'],
    te_err_core=core['Te_err'], te_err_edge=edge['Te_err'],
    Rsep_eqh=eqh_result['Rsep_eqh'], Rsep_spline=spline_result['Rsep'],
    Rsep_mtanh=mtanh_result['Rsep'], knot_pos=knots,
    nshot=shot_id, interval=interval, equilibrium=equilibrium_name,
    r_zoom=2.12, directory=str(output_directory),
    fig_name=make_fig_name(shot_id, equilibrium_name, len(knots), interval, 'elm_cycle_summary'),
    save_fig=save_figures, show=False, close=not show_figures,x_mins = 2.13, x_maxs=2.14
)
ax.flatten()[9].set_xlim(2.135, 2.145)
ax.flatten()[7].set_xlim(2.135, 2.145)
ax.flatten()[7].set_ylim(0,0.3)
ax.flatten()[5].set_xlim(2.135, 2.165)
fig.show()

# Several plots to compare to Davide's FusionFit and data

Error perturbation is intentionally not included yet. The fits use the measured VTA uncertainties after the normal preprocessing filters.

In [ ]:
# Notebook-only comparison with Davide's Te and Ne references.
import os
import re
import matplotlib.pyplot as plt
from ts_profile_fitter.plotting import plot_profile_fit
reference_folder = '/shares/departments/AUG/users/dsil/fusionfit/sav/data/sep/32959/'
reference_filenames = {
    'Te_fit-mtanh': '32959_4.300_AUGD_EQH_1_Te_fit-mtanh.dat',
    'Te_edgeTS': '32959_4.300_AUGD_EQH_1_Te_edgeTS.dat',
    'Te_coreTS': '32959_4.300_AUGD_EQH_1_Te_coreTS.dat',
    'rhop_Te_edge': '32959_4.300_AUGD_EQH_1_rhop_Te_edgeTS.dat',
    'Ne_fit-mtanh': '32959_4.300_AUGD_EQH_1_ne_fit-mtanh.dat',
    'Ne_edgeTS': '32959_4.300_AUGD_EQH_1_ne_edgeTS.dat',
    'Ne_coreTS': '32959_4.300_AUGD_EQH_1_ne_coreTS.dat',
}

def read_davide_profile(label, filename):
    path = os.path.join(reference_folder, filename)
    with open(path, 'r') as file:
        header_tokens = file.readline().lstrip('#').strip().split()
    columns = [f'{header_tokens[i]} {header_tokens[i + 1]}'
               for i in range(0, len(header_tokens), 2)]
    return pd.read_csv(path, sep=r'\s+', comment='#', names=columns)

reference_profiles = {
    label: read_davide_profile(label, filename)
    for label, filename in reference_filenames.items()
}

# Select only edge Te values whose complete TS rows were removed by Ne filtering.
removed_edge_ne = preprocess_metadata['removed_rows'].loc[
    (
        (preprocess_metadata['removed_rows']['system'] == 'edge')
        & preprocess_metadata['removed_rows']['filter_reason'].str.contains(
            'Ne_outlier', na=False
        )
    )
].copy()
print(f'Edge Te points rejected by Ne edge filtering: {len(removed_edge_ne)}')

print('Davide reference folder:', reference_folder)
for label, profile in reference_profiles.items():
    print(f'{label}: {len(profile)} rows; columns={list(profile.columns)}')
    print(profile.describe().loc[['min', 'max']])

# The reference files store Te in keV and Ne in 10^19 m^-3.
davide_fit = reference_profiles['Te_fit-mtanh']
davide_x = davide_fit['Rmaj [m]'].to_numpy(float)
davide_te = 1e3 * davide_fit['fit_data [keV]'].to_numpy(float)
davide_te_err = 1e3 * davide_fit['dfit_data [keV]'].to_numpy(float)
davide_Ne = 1e19 * reference_profiles['Ne_fit-mtanh']['fit_data [10^19'].to_numpy(float)
davide_Ne_err = 1e19 * reference_profiles['Ne_fit-mtanh']['m^-3] dfit_data'].to_numpy(float)

In [ ]:
# Use a 1 micrometer search grid to match the legacy separatrix crossing.
davide_x_dense = np.arange(2.0, davide_x.max() - 0.02, 1e-6)

In [ ]:
# Compare Davide's source TS points with the current core/edge data separately.
def compare_source_to_current(reference_profile, current_profile):
    reference_r = reference_profile['Rmaj [m]'].to_numpy(float)
    reference_te = 1e3 * reference_profile.iloc[:, 2].to_numpy(float)
    current_r = current_profile['r_midplane'].to_numpy(float)
    current_te = current_profile['Te'].to_numpy(float)

    nearest_index = np.abs(reference_r[:, None] - current_r[None, :]).argmin(axis=1)
    nearest_r = current_r[nearest_index]
    nearest_te = current_te[nearest_index]
    radial_difference = nearest_r - reference_r
    te_difference = nearest_te - reference_te
    matched = np.abs(radial_difference) <= 0.005

    return {
        # In these flattened TS tables, one row corresponds to one measured point.
        'Davide_source_points_rows': int(reference_r.size),
        'current_points_rows': int(current_r.size),
        'Davide_R_range_m': (float(reference_r.min()), float(reference_r.max())),
        'current_R_midplane_range_m': (float(current_r.min()), float(current_r.max())),
        'Davide_Te_range_eV': (float(reference_te.min()), float(reference_te.max())),
        'current_Te_range_eV': (float(current_te.min()), float(current_te.max())),
        'nearest_R_difference_mean_abs_m': float(np.mean(np.abs(radial_difference))),
        'nearest_R_difference_max_abs_m': float(np.max(np.abs(radial_difference))),
        'matches_within_5mm': int(np.sum(matched)),
        'Te_difference_mean_eV_within_5mm': (
            float(np.mean(te_difference[matched])) if np.any(matched) else None
        ),
        'Te_difference_mae_eV_within_5mm': (
            float(np.mean(np.abs(te_difference[matched]))) if np.any(matched) else None
        ),
        'Te_difference_rmse_eV_within_5mm': (
            float(np.sqrt(np.mean(te_difference[matched] ** 2))) if np.any(matched) else None
        ),
    }

core_edge_comparison = {
    'core': compare_source_to_current(reference_profiles['Te_coreTS'], core),
    'edge': compare_source_to_current(reference_profiles['Te_edgeTS'], edge),
}
print('Davide source vs current preprocessed core/edge points:')
print(core_edge_comparison)


## Compare my fit on Davide's Data, my fit on my Data and his fit on his Data

In [ ]:
# Compare three fits: my fit on my data, my fit on Davide's data, and Davide's saved paper fit
current_r = np.concatenate([
    core['r_midplane'].to_numpy(float),
    edge['r_midplane'].to_numpy(float),
])
current_te = np.concatenate([
    core['Te'].to_numpy(float),
    edge['Te'].to_numpy(float),
])
current_te_err = np.concatenate([
    core['Te_err'].to_numpy(float),
    edge['Te_err'].to_numpy(float),
])
current_order = np.argsort(current_r)
current_r = current_r[current_order]
current_te = current_te[current_order]
current_te_err = current_te_err[current_order]

reference_te_data = np.concatenate([
    1e3 * reference_profiles['Te_coreTS'].iloc[:, 2].to_numpy(float),
    1e3 * reference_profiles['Te_edgeTS'].iloc[:, 2].to_numpy(float),
])
reference_te_err_data = np.concatenate([
    1e3 * reference_profiles['Te_coreTS'].iloc[:, 3].to_numpy(float),
    1e3 * reference_profiles['Te_edgeTS'].iloc[:, 3].to_numpy(float),
])
reference_r_data = np.concatenate([
    reference_profiles['Te_coreTS']['Rmaj [m]'].to_numpy(float),
    reference_profiles['Te_edgeTS']['Rmaj [m]'].to_numpy(float),
])
reference_order = np.argsort(reference_r_data)
reference_r_data = reference_r_data[reference_order]
reference_te_data = reference_te_data[reference_order]
reference_te_err_data = reference_te_err_data[reference_order]

# Fit the two datasets independently with my routine.
current_fit_on_current = modified_tanh(current_r, *np.asarray(mtanh_te[0], dtype=float))
current_fit_x = np.linspace(current_r.min(), current_r.max(), 500)
current_fit_on_current_grid = modified_tanh(
    current_fit_x, *np.asarray(mtanh_te[0], dtype=float)
)
davide_mtanh = fit_modified_tanh_profile(
    reference_r_data,
    reference_te_data,
    reference_te_err_data,
    deg_tail=mtanh_config['Te']['deg_tail'],
    deg_top=mtanh_config['Te']['deg_top'],
    y_scale=mtanh_config['Ne']['y_scale'],
    fix_a_tail=mtanh_config['Te']['fix_a_tail'],
)
reference_r_data_fit = np.linspace(current_r.min(), current_r.max(), 500)
my_fit_on_davides_data = modified_tanh(
    reference_r_data_fit, *np.asarray(davide_mtanh[0], dtype=float)
)
# Keep the saved Davide paper fit separate from the my refit.
reference_fit_on_reference = np.interp(reference_r_data, davide_x, davide_te)
davide_te_at_2_11 = float(np.interp(2.11, davide_x, davide_te))
print(f'Davide saved mtanh at R_midplane=2.11 m: {davide_te_at_2_11:.6f} eV')
print(f'Davide data refit parameters: {davide_mtanh[0]}')

def fit_metrics(data, fit, errors):
    residual = data - fit
    valid = np.isfinite(residual) & np.isfinite(errors) & (errors > 0)
    return {
        'points': int(np.sum(valid)),
        'bias_eV': float(np.mean(residual[valid])),
        'mae_eV': float(np.mean(np.abs(residual[valid]))),
        'rmse_eV': float(np.sqrt(np.mean(residual[valid] ** 2))),
        'reduced_chi2': float(np.mean((residual[valid] / errors[valid]) ** 2)),
    }

three_fit_comparison = {
    'my_data_my_fit': fit_metrics(current_te, current_fit_on_current, current_te_err),
    'Davide_data_my_routine_fit': fit_metrics(reference_te_data, reference_r_data, reference_te_err_data),
    'Davide_data_saved_paper_fit': fit_metrics(reference_te_data, reference_fit_on_reference, reference_te_err_data),
}
print('Three Te fit comparisons:')
print(three_fit_comparison)

from ipfnpytools.plot import plots as ipfn_plots

fig_three_fits, ax_three_fits = ipfn_plots(
    x=[current_r, reference_r_data, current_fit_x, reference_r_data_fit, davide_x],
    y=[
        current_te,
        reference_te_data,
        current_fit_on_current_grid,
        my_fit_on_davides_data,
        davide_te,
    ],
    z=[None, None, None, None, None],
    ey=[current_te_err, reference_te_err_data, None, None, None],
    x_labels='R_midplane [m]',
    y_labels='Te [eV]',
    labels=[
        'My data',
        "Davide's data",
        'My fit on my data grid',
        "My routine fit on Davide's data",
        "Davide's saved mtanh fit"
    ],
    linestyles=['', '', '-', '--', ':'],
    markers=['o', 'x', None, None, None],
    markersizes=[6, 6, None, None, None],
    boxes=0,
    write_legends=True,
    x_mins=2.11,
    x_maxs=2.19,
    y_mins=-100,
    y_maxs=1200,
    h_factor=1.5,
    title=f"Shot {shot_id}, [{interval['t_begin']:.3f}, {interval['t_end']:.3f}] s: My vs Davide's data overlay and fits",
    fig_name=make_fig_name(
        shot_id, equilibrium_name, len(knots), interval, 'te_my_vs_davide_data_overlay_fits'
    ),
    save_figure_kw={'directory': str(output_directory)},
    plot_kw=[
        {'ecolor': 'yellow', 'alpha': 0.45, 'color': 'yellow'},
        {'ecolor': 'tab:blue', 'alpha': 0.35, 'color': 'tab:blue'},
        {'color': 'tab:blue', 'linewidth': 2},
        {'color': 'tab:orange', 'linewidth': 2},
        {'color': 'black', 'linewidth': 2},
    ],
    close=False,
    save_fig=save_figures,
    show=show_figures,
 )

## Metrics and Several fits

In [ ]:
ds_out_ondrej, knots_ondrej, p_ne_ondrej, pcov_ne_ondrej, p_te_ondrej, pcov_te_ondrej = fit_mbsplt_ne_Te_TS(
        spline_input,
        dR_fine=active_spline_config['dR_fine'],
        sigma_conv=active_spline_config['sigma_conv'],
        R_knots=knots,
        dR_channel_max=active_spline_config['dR_channel_max'],
        max_step_scale=active_spline_config['max_step_scale'],
        use_errors=active_spline_config['use_errors'],
        ne_scale=active_spline_config['ne_scale'],
        Te_scale=active_spline_config['Te_scale'],
    )

In [ ]:
# Common fit-quality metrics on the experimental points.
from ts_profile_fitter.fit_metrics import compute_fit_metrics
from ts_profile_fitter.fitting import count_modified_tanh_parameters

mtanh_n_parameters = {
    quantity: count_modified_tanh_parameters(
        deg_tail=config['deg_tail'],
        deg_top=config['deg_top'],
        fix_x0=config.get('fix_x0'),
        fix_w=config.get('fix_w'),
        fix_a_tail=config.get('fix_a_tail'),
    )
    for quantity, config in active_mtanh_config.items()
}
fit_metric_rows = []
model_predictions = {
    'my mtanh': {
        'Te': lambda x: modified_tanh(x, *np.asarray(mtanh_te[0], dtype=float)),
        'Ne': lambda x: modified_tanh(x, *np.asarray(mtanh_ne[0], dtype=float)),
        'n_parameters': mtanh_n_parameters,
    },
    'IPFN spline': {
        'Te': lambda x: np.interp(x, ds_out['R_mid_spl'], ds_out['Te_spl']),
        'Ne': lambda x: np.interp(x, ds_out['R_mid_spl'], ds_out['ne_spl']),
        'n_parameters': {'Te': len(p_te), 'Ne': len(p_ne)},
    },
    'Ondrej spline': {
        'Te': lambda x: np.interp(x, ds_out_ondrej['R_mid_spl'], ds_out_ondrej['Te_spl']),
        'Ne': lambda x: np.interp(x, ds_out_ondrej['R_mid_spl'], ds_out_ondrej['ne_spl']),
        'n_parameters': {'Te': len(p_te_ondrej), 'Ne': len(p_ne_ondrej)},
    },
    'Davide\'s saved fit': {
        'Te': lambda x: np.interp(x, reference_profiles['Te_fit-mtanh']['Rmaj [m]'].to_numpy(float), 1e3 * reference_profiles['Te_fit-mtanh'].iloc[:, 2].to_numpy(float)),
        'Ne': lambda x: np.interp(x, reference_profiles['Ne_fit-mtanh']['Rmaj [m]'].to_numpy(float), 1e19 * reference_profiles['Ne_fit-mtanh'].iloc[:, 2].to_numpy(float)),
        'n_parameters': {'Te': None, 'Ne': None},
    }
}

for quantity, x_values, y_values, y_errors in [
    ('Te', x_te, y_te, yerr_te),
    ('Ne', x_ne, y_ne, yerr_ne),
]:
    for region_name, region_mask in [
        ('all', np.ones(len(x_values), dtype=bool)),
        ('core', mapped['system'].to_numpy() == 'core'),
        ('edge', mapped['system'].to_numpy() == 'edge'),
    ]:
        for model_name, model in model_predictions.items():
            metrics = compute_fit_metrics(
                y_observed=y_values[region_mask],
                y_fitted=model[quantity](x_values[region_mask]),
                y_error=y_errors[region_mask],
                n_parameters=model['n_parameters'][quantity],
            )
            fit_metric_rows.append({
                'dataset': 'My Dataset',
                'model': model_name,
                'quantity': quantity,
                'region': region_name,
                'n_parameters': model['n_parameters'][quantity],
                **metrics,
            })


def add_davide_reference_metrics(quantity, fit_key, core_key, edge_key, scale):
    profiles = [reference_profiles[core_key], reference_profiles[edge_key]]
    region_arrays = []
    for region_name, profile in zip(('core', 'edge'), profiles):
        region_arrays.append((
            region_name,
            profile['Rmaj [m]'].to_numpy(float),
            scale * profile.iloc[:, 2].to_numpy(float),
            scale * profile.iloc[:, 3].to_numpy(float),
        ))

    all_x = np.concatenate([region_arrays[0][1], region_arrays[1][1]])
    all_y = np.concatenate([region_arrays[0][2], region_arrays[1][2]])
    all_errors = np.concatenate([region_arrays[0][3], region_arrays[1][3]])
    region_arrays.append(('all', all_x, all_y, all_errors))

    saved_fit = reference_profiles[fit_key]
    saved_x = saved_fit['Rmaj [m]'].to_numpy(float)
    saved_y = scale * saved_fit.iloc[:, 2].to_numpy(float)
    mtanh_parameters = mtanh_te[0] if quantity == 'Te' else mtanh_ne[0]
    mtanh_fit = lambda x: modified_tanh(x, *np.asarray(mtanh_parameters, dtype=float))

    for region_name, x_values, y_values, y_errors in region_arrays:
        for model_name, y_fitted, n_parameters in [
            ("Davide's saved fit", np.interp(x_values, saved_x, saved_y), None),
            ('my mtanh', mtanh_fit(x_values), mtanh_n_parameters[quantity]),
        ]:
            metrics = compute_fit_metrics(
                y_observed=y_values,
                y_fitted=y_fitted,
                y_error=y_errors,
                n_parameters=n_parameters,
            )
            fit_metric_rows.append({
                'dataset': f"Davide's {quantity} data",
                'model': model_name,
                'quantity': quantity,
                'region': region_name,
                'n_parameters': n_parameters,
                **metrics,
                'n_points': len(y_values),
            })


add_davide_reference_metrics('Te', 'Te_fit-mtanh', 'Te_coreTS', 'Te_edgeTS', 1e3)
add_davide_reference_metrics('Ne', 'Ne_fit-mtanh', 'Ne_coreTS', 'Ne_edgeTS', 1e19)

fit_metrics_table = pd.DataFrame(fit_metric_rows)
metric_columns = [
    'dataset', 'model', 'quantity', 'region', 'n_points', 'n_parameters', 'weighted_sse', 'reduced_chi2',
]
print('Te fit metrics')
display(fit_metrics_table.query("quantity == 'Te'").query('region == "all"')[metric_columns])
print('Ne fit metrics')
display(fit_metrics_table.query("quantity == 'Ne'").query('region == "all"')[metric_columns])

## dW/dt exploration

In [ ]:
# Explore Davide's fast-Wmhd and NME2025 reference data before the power-balance sweep.
from pathlib import Path
import aug_sfutils as sf

nme_directory = Path('/shares/departments/AUG/users/dsil/workspace/separatrix/db/AUG/NME2025_dataset')
fast_wmhd_path = Path('/shares/departments/AUG/users/dsil/workspace/separatrix/db/AUG/Wmhd_fast/eqdsk_32959_4.1-4.5_wmhd.dat')
nme_general = pd.read_csv(nme_directory / 'DB_PPCFKallenbach2018_AUG_general.csv', low_memory=False)
nme_standard = pd.read_csv(nme_directory / 'DB_PPCFKallenbach2018_AUG_sep_standard_settings.csv', low_memory=False)
nme_general_32959 = nme_general.loc[nme_general['shot'].eq(shot_id)].copy()
nme_standard_32959 = nme_standard.loc[nme_standard['shot'].eq(shot_id)].copy()

fast_wmhd_array = np.loadtxt(fast_wmhd_path, usecols=(0, 1))
fast_wmhd_time = fast_wmhd_array[:, 0]
fast_wmhd_values = fast_wmhd_array[:, 1]

elm_shotfile = sf.SFREAD(shot_id, 'ELM')
elm_signal = np.asarray(elm_shotfile.getobject(
    't_begELM', tbeg=interval['t_begin'], tend=interval['t_end'] 
), dtype=float).ravel()
elm_signal = elm_signal[np.isfinite(elm_signal)]

tot_shotfile = sf.SFREAD(shot_id, 'TOT')
tot_dwdt_signal = np.asarray(tot_shotfile.getobject(
    'dWmhd/dt', tbeg=interval['t_begin'], tend=interval['t_end']
), dtype=float).ravel()
tot_dwdt_signal = tot_dwdt_signal[np.isfinite(tot_dwdt_signal)]

print(f'Fast-Wmhd file: {fast_wmhd_path}')
print(f'Fast-Wmhd samples: {len(fast_wmhd_values)}; dt = {np.median(np.diff(fast_wmhd_time)):.6g} s')
print(f'ELM markers: {len(elm_signal)}; TOT dW/dt samples: {len(tot_dwdt_signal)}')
print('NME2025 general rows for this shot:')
display(nme_general_32959)
print('NME2025 separatrix rows for this shot:')
display(nme_standard_32959)

fig_reference, reference_axes = plt.subplots(2, 1, figsize=(11, 7), sharex=False)
reference_axes[0].plot(fast_wmhd_time, fast_wmhd_values / 1e3, color='tab:blue')
reference_axes[0].set_title(f'Shot {shot_id}: Davide fast Wmhd file')
reference_axes[0].set_ylabel('Wmhd [kJ]')
reference_axes[0].grid(True, alpha=0.25)
reference_axes[1].plot(fast_wmhd_time, fast_wmhd_values / 1e3, color='tab:blue', label='fast Wmhd')
for elm_time in elm_signal:
    reference_axes[1].axvline(elm_time, color='tab:red', alpha=0.3, linewidth=0.8)
reference_axes[1].set_xlim(interval['t_begin'], interval['t_end'])
reference_axes[1].set_xlabel('Time [s]')
reference_axes[1].set_ylabel('Wmhd [kJ]')
reference_axes[1].set_title('Fast Wmhd with ELM markers')
reference_axes[1].grid(True, alpha=0.25)
reference_axes[1].legend()
fig_reference.tight_layout()
plt.show()

In [ ]:
# dW/dt exploration for Davide's fitted profile.
from aug_sfutils import sfread
from get_functions import dWdt_Field
from scipy.interpolate import interp1d
from ipfnpytools.plot import plots as ipfn_plots

shared_physics_inputs = load_physics_inputs(
    shot_id,
    interval,
    equilibrium=equilibrium_name,
    dWdt_method='paper',
    elm_fraction=(elm_cycle_start_fraction, elm_cycle_end_fraction),
)
shared_power = dict(shared_physics_inputs['power_balance'])
shared_geometry = dict(shared_physics_inputs['geometry'])
shared_geometry['L_par'] = physics_inputs['geometry']['L_par']

t_begin = float(interval['t_begin'])
t_end = float(interval['t_end'])
phase_begin = float(elm_cycle_start_fraction)
phase_end = float(elm_cycle_end_fraction)

# Use only ELM markers inside the requested interval; no padding is applied.
elm_signal = np.asarray(
    elm_shotfile.getobject('t_begELM', tbeg=t_begin, tend=t_end),
    dtype=float,
).ravel()
elm_signal = np.unique(np.sort(elm_signal[np.isfinite(elm_signal)]))


def _uniform_wmhd(time, values, label):
    time = np.asarray(time, dtype=float).ravel()
    values = np.asarray(values, dtype=float).ravel()
    finite = np.isfinite(time) & np.isfinite(values)
    time, values = time[finite], values[finite]
    order = np.argsort(time)
    time, values = time[order], values[order]
    time, unique = np.unique(time, return_index=True)
    values = values[unique]
    if time.size < 2 or np.any(np.diff(time) <= 0.0):
        raise RuntimeError(f'{label} Wmhd timebase is invalid.')
    dt = float(np.mean(np.diff(time)))
    n_samples = int(np.ceil((time[-1] - time[0]) / dt))
    if not np.isfinite(dt) or dt <= 0.0 or n_samples < 2:
        raise RuntimeError(f'{label} Wmhd timebase is invalid.')
    uniform_time = np.linspace(time.min(), time.max(), n_samples)
    uniform_values = interp1d(time, values)(uniform_time)
    return uniform_time, uniform_values


def _field_dwdt(time, values, label):
    dwdt_mw, dwdt_err_mw = dWdt_Field(
        time, values, elm_signal, shot_id, phase_begin, phase_end,
    )
    return {
        'label': label,
        'dWdt_MW': float(dwdt_mw),
        'dWdt_err_MW': float(dwdt_err_mw),
    }

# Mike Dunne's fast-Wmhd file.
mike_time, mike_wmhd = _uniform_wmhd(
    fast_wmhd_time, fast_wmhd_values, 'Mike Dunne fast',
)
mike_dwdt = _field_dwdt(
    mike_time, mike_wmhd, 'Field method - Mike Dunne fast Wmhd',
)

# GQH Wmhd from the equilibrium shotfile, restricted to the exact interval.
eq = ts_data['equ']
eqexp = getattr(eq, 'exp', None)
eqdia = getattr(eq, 'diag', equilibrium_name)
eqed = getattr(eq, 'ed', None)
gqh = sfread.SFREAD('GQH', shot_id, exp=eqexp, diag=eqdia, ed=eqed)
wmhd = gqh.getobject('Wmhd', tbeg=t_begin, tend=t_end)
wmhd_t = gqh.gettimebase('Wmhd', tbeg=t_begin, tend=t_end)
gqh_time, gqh_wmhd = _uniform_wmhd(wmhd_t, wmhd, 'GQH')
gqh_dwdt = _field_dwdt(
    gqh_time, gqh_wmhd, 'Field method - GQH Wmhd',
)

dwdt_inter_elm_mw = mike_dwdt['dWdt_MW']
dwdt_inter_elm_err_mw = mike_dwdt['dWdt_err_MW']
dwdt_mike_fast_mw = mike_dwdt['dWdt_MW']
dwdt_mike_fast_err_mw = mike_dwdt['dWdt_err_MW']
dwdt_gqh_mw = gqh_dwdt['dWdt_MW']
dwdt_gqh_err_mw = gqh_dwdt['dWdt_err_MW']

print(f"{mike_dwdt['label']}: {dwdt_mike_fast_mw:.6g} +/- {dwdt_mike_fast_err_mw:.4g} MW")
print(f"{gqh_dwdt['label']}: {dwdt_gqh_mw:.6g} +/- {dwdt_gqh_err_mw:.4g} MW")

dwdt_values = {
    'tot': float(np.nanmean(tot_dwdt_signal)) / 1e6,
    'paper': 0.25 * shared_power['P_abs'] / 1e6,
    'mike_fast': dwdt_mike_fast_mw,
    'gqh': dwdt_gqh_mw,
    '0.4*P_abs': 0.4 * shared_power['P_abs'] / 1e6,
}
dwdt_labels = {
    'tot': 'TOT dW/dt',
    'paper': 'Paper dW/dt = 0.25 P_abs',
    'mike_fast': mike_dwdt['label'],
    'gqh': gqh_dwdt['label'],
    '0.4*P_abs': 'dW/dt = 0.4 P_abs',
}
dwdt_errors = {
    'mike_fast': dwdt_mike_fast_err_mw,
    'gqh': dwdt_gqh_err_mw,
}

dwdt_results = {}
dwdt_summary_rows = []
for case_name, dwdt_value_mw in dwdt_values.items():
    case_power = dict(shared_power)
    case_power['dWdt'] = dwdt_value_mw * 1e6
    case_power['dWdt_err'] = dwdt_errors.get(case_name, np.nan) * 1e6
    result = compute_physics_separatrix(
        davide_x,
        davide_te,
        Ne=davide_Ne,
        power_balance=case_power,
        geometry=shared_geometry,
        x_grid=davide_x_dense,
        fit_method=f'davide-{case_name}',
    )
    dwdt_results[case_name] = result
    dwdt_summary_rows.append({
        'method': case_name,
        'label': dwdt_labels[case_name],
        'P_abs_MW': shared_power['P_abs'] / 1e6,
        'P_rsep_MW': shared_power['P_rsep'] / 1e6,
        'dWdt_MW': dwdt_value_mw,
        'dWdt_err_MW': dwdt_errors.get(case_name, np.nan),
        'P_SOL_MW': result['P_SOL'] / 1e6,
        'P_e_SOL_MW': result['P_e_SOL'] / 1e6,
        'Rsep_m': result['Rsep'],
    })

if nme_standard_32959.empty:
    raise ValueError(f'No NME2025 separatrix row found for shot {shot_id}.')
nme_reference = nme_standard_32959.iloc[0]
davide_csv_pabs = float(nme_reference['Pabs [MW]'])
davide_csv_prsep = float(nme_reference['Prsep [MW]'])
davide_csv_dwdt = float(nme_reference['dW/dt [MW]'])
davide_csv_psol = davide_csv_pabs - davide_csv_prsep - davide_csv_dwdt
davide_csv_pesol = davide_csv_psol * 0.8 * 0.65 * 0.8
dwdt_summary_rows.append({
    'method': 'davide_csv',
    'label': 'Davide CSV standard settings',
    'P_abs_MW': davide_csv_pabs,
    'P_rsep_MW': davide_csv_prsep,
    'dWdt_MW': davide_csv_dwdt,
    'dWdt_err_MW': np.nan,
    'P_SOL_MW': davide_csv_psol,
    'P_e_SOL_MW': davide_csv_pesol,
    'Rsep_m': float(nme_reference['Rsep [m]']),
})
dwdt_summary = pd.DataFrame(dwdt_summary_rows)
display(dwdt_summary)

In [ ]:
shared_power

In [ ]:
davide_power = dict(shared_power)

In [ ]:
# Compare the power-balance profiles for every dW/dt case.
plot_cases = list(dwdt_results)
plot_x = [dwdt_results[name]['R_grid'] for name in plot_cases] + [dwdt_results['mike_fast']['R_grid']]
plot_y = [
    dwdt_results[name]['q_par_P_SOL_grid'] / 1e9
    for name in plot_cases
] + [dwdt_results['mike_fast']['q_par_SH_grid'] / 1e9]
plot_labels = [
    f"{dwdt_labels[name]} ({dwdt_values[name]:.3g} +/- {dwdt_errors.get(name, np.nan):.3g} MW)"
    if name in dwdt_errors
    else dwdt_labels[name]
    for name in plot_cases
] + ['q_cond']
plot_linestyles = ['-'] * (len(plot_cases)+1)
plot_styles = [{'linewidth': 2.0} for _ in plot_cases] + [{'linewidth': 2.0}]

fig_dwdt, ax_dwdt = ipfn_plots(
    x=plot_x,
    y=plot_y,
    z=[None] * len(plot_x),
    ey=[None] * len(plot_x),
    labels=plot_labels,
    linestyles=plot_linestyles,
    markers=[None] * len(plot_x),
    boxes=0,
    write_legends=True,
    x_labels='R_midplane [m]',
    y_labels=r'$q_{SOL}$ [GW/m$^2$]',
    title=f'Shot {shot_id}: dW/dt source comparison',
    x_mins=2.10,
    x_maxs=2.165,
    y_mins=0.0,
    y_maxs=0.5,
    h_factor=1.7,
    fig_name=make_fig_name(
        shot_id, equilibrium_name, len(knots), interval,
        'davide_dwdt_source_comparison',
    ),
    save_figure_kw={'directory': str(output_directory)},
    plot_kw=plot_styles,
    close=False,
    save_fig=False,
    show=False,
)

rsep_colors = {
    name: color
    for name, color in zip(
        plot_cases,
        ['tab:blue', 'tab:orange', 'tab:green', 'tab:red', 'tab:purple'],
    )
}
for case_name, result in dwdt_results.items():
    ax_dwdt.axvline(
        result['Rsep'],
        color=rsep_colors[case_name],
        linestyle=':',
        linewidth=1.5,
        alpha=0.9,
        label=f"{dwdt_labels[case_name]} Rsep = {result['Rsep']:.4f} m",
    )
ax_dwdt.axvline(
    float(nme_reference['Rsep [m]']),
    color='black',
    linestyle='-.',
    linewidth=1.5,
    label=f"Davide's Paper Rsep = {nme_reference['Rsep [m]']:.4f} m",
)
ax_dwdt.legend(loc='best', fontsize=8)
ax_dwdt.grid(True, alpha=0.25)
#fig_dwdt.tight_layout()

if save_figures:
    fig_dwdt.savefig(
        output_directory / make_fig_name(
            shot_id, equilibrium_name, len(knots), interval,
            'davide_dwdt_source_comparison',
        ),
        bbox_inches='tight',
    )
if show_figures:
    plt.show()
else:
    plt.close(fig_dwdt)


In [ ]:
physics_inputs['geometry']

In [ ]:
repo_case.keys()

In [ ]:
# Systematic comparison of the quantities entering the conduction-limited q_SOL model.
base = '/shares/departments/AUG/users/dsil/workspace/separatrix/db/AUG/NME2025_dataset'
standard = pd.read_csv(f'{base}/DB_PPCFKallenbach2018_AUG_sep_standard_settings.csv')
general = pd.read_csv(f'{base}/DB_PPCFKallenbach2018_AUG_general.csv')
lc_database = pd.read_csv(f'{base}/DB_PPCFKallenbach2018_AUG_Lc_1mm_all.csv')
davide_row = standard.loc[standard['shot'].eq(shot_id)].iloc[0]
general_row = general.loc[general['shot'].eq(shot_id)].iloc[0]
lc_row = lc_database.loc[lc_database['shot'].eq(shot_id)].iloc[0]
repo_case = dwdt_results['mike_fast']

f_e, f_out, f_cond = 0.8, 0.65, 0.8
f_product = f_e * f_out * f_cond

def _number(value):
    return float(value) if pd.notna(value) else np.nan

def _difference(davide_value, repo_value):
    if not np.isfinite(davide_value) or not np.isfinite(repo_value):
        return np.nan
    return repo_value - davide_value

def _comparison(category, quantity, unit, davide_value, repo_value, equation):
    return {
        'category': category,
        'quantity': quantity,
        'unit': unit,
        'Davide': davide_value,
        'Repo': repo_value,
        'Repo - Davide': _difference(davide_value, repo_value),
        'equation / source': equation,
    }

# The repo values below come directly from the evaluated pysepest state.
# No interpolation is needed: repo_case already contains the scalar model state.
p_abs_d = _number(davide_row['Pabs [MW]'])
p_rsep_d = _number(davide_row['Prsep [MW]'])
dwdt_d = _number(davide_row['dW/dt [MW]'])
p_sol_d = p_abs_d - p_rsep_d - dwdt_d
p_abs_r = float(shared_power['P_abs']) / 1e6
p_rsep_r = float(shared_power['P_rsep']) / 1e6
p_sol_r = float(repo_case['P_SOL']) / 1e6
p_e_sol_d = p_sol_d * f_product
p_e_sol_r = float(repo_case['P_e_SOL']) / 1e6

# Direct outer-midplane geometry and connection length.
r_omp_d = _number(davide_row['Rout [m]'])
bp_omp_d = _number(davide_row['Bpout [T]'])
bt_omp_d = _number(davide_row['Btout [T]'])
l_sol_d = _number(lc_row['Lc [m]'])
r_omp_r = float(repo_case['R'])
bp_omp_r = float(repo_case['B_pol'])
bt_omp_r = float(repo_case['B_tor'])
l_sol_r = float(repo_case['L_par'])

# Direct separatrix/model values from the result dictionary.
rsep_d = _number(davide_row['Rsep [m]'])
rsep_r = float(repo_case['Rsep'])
te_sep_d = _number(davide_row['Tsep [eV]'])
te_sep_r = float(repo_case['Te_sep'])
ne_sep_d = _number(davide_row['nsep [1e19 m^-3]'])
ne_sep_r = (float(repo_case['Ne_sep']) / 1e19
             if np.isfinite(repo_case.get('Ne_sep', np.nan)) else np.nan)
lam_te_d = _number(davide_row['lamT [mm]']) / 1e3
lam_te_r = float(repo_case['lam_Te'])
lam_q_d = 2.0 / 7.0 * lam_te_d
lam_q_r = float(repo_case['lam_q'])
zeff_d = _number(general_row['Zeff'])
zeff_r = float(repo_case['Zeff'])
k0_d = 2600.0 / (0.672 + 0.076 * zeff_d**0.5 + 0.252 * zeff_d)
k0_r = float(repo_case['kappa_0'])
q_p_sol_d = _number(davide_row['q_Psol [MW]'])
q_cond_d = _number(davide_row['q_SH [MW]'])
q_p_sol_r = float(repo_case['q_par_P_SOL']) / 1e6
q_cond_r = float(repo_case['q_par_SH']) / 1e6

# A_SOL is also already evaluated directly by the model.
a_sol_d_geom = 2.0 * np.pi * r_omp_d * lam_q_d * abs(bp_omp_d / bt_omp_d)
a_sol_d_power = p_e_sol_d / q_p_sol_d
a_sol_r = float(repo_case['A_SOL'])
a_sol_r_power = (float(repo_case['P_e_SOL']) / float(repo_case['q_par_P_SOL']))

comparison_rows = [
    _comparison('power balance', 'P_abs', 'MW', p_abs_d, p_abs_r, 'TTH P_NET'),
    _comparison('power balance', 'P_rsep', 'MW', p_rsep_d, p_rsep_r, 'BPT Pr_sep'),
    _comparison('power balance', 'dW/dt', 'MW', dwdt_d, dwdt_mike_fast_mw, 'GQH Wmhd fast Field method'),
    _comparison('power balance', 'P_SOL', 'MW', p_sol_d, p_sol_r, 'P_abs - P_rsep - dW/dt'),
    _comparison('power balance', 'P_e_SOL', 'MW', p_e_sol_d, p_e_sol_r, 'P_SOL * f_e * f_out * f_cond'),
    _comparison('outer-midplane geometry', 'R_omp', 'm', r_omp_d, r_omp_r, 'Rout [m] / equilibrium rho=1, theta=0'),
    _comparison('outer-midplane geometry', 'B_p_omp', 'T', bp_omp_d, bp_omp_r, 'Bpout [T] = sqrt(Br_out^2 + Bz_out^2)'),
    _comparison('outer-midplane geometry', 'B_t_omp', 'T', bt_omp_d, bt_omp_r, 'Btout [T] / equilibrium rz2brzt'),
    _comparison('connection length', 'L_sol', 'm', l_sol_d, l_sol_r, 'Davide Lc [m] / repo L_par'),
    _comparison('separatrix', 'Rsep', 'm', rsep_d, rsep_r, 'CSV Rsep / pysepest Rsep'),
    _comparison('separatrix', 'Te_sep', 'eV', te_sep_d, te_sep_r, 'CSV Tsep / repo Te_sep'),
    _comparison('separatrix', 'ne_sep', '1e19 m^-3', ne_sep_d, ne_sep_r, 'CSV nsep / repo Ne_sep'),
    _comparison('fall-off lengths', 'lambda_Te', 'mm', lam_te_d * 1e3, lam_te_r * 1e3, 'Te / -dTe/dR'),
    _comparison('fall-off lengths', 'lambda_q', 'mm', lam_q_d * 1e3, lam_q_r * 1e3, '2/7 * lambda_Te'),
    _comparison('conduction model', 'q_P_SOL', 'MW/m^2', q_p_sol_d, q_p_sol_r, 'P_e_SOL / A_SOL'),
    _comparison('conduction model', 'q_cond', 'MW/m^2', q_cond_d, q_cond_r, '2/7*q_distribution*k0*Te^3.5/L_sol'),
    _comparison('conduction model', 'q_cond - q_P_SOL', 'MW/m^2', q_cond_d - q_p_sol_d, q_cond_r - q_p_sol_r, 'separatrix condition'),
    _comparison('SOL area', 'A_SOL', 'm^2', a_sol_d_geom, a_sol_r, '2*pi*R*lambda_q*abs(B_p/B_t)'),
    _comparison('SOL area', 'A_SOL from P/q', 'm^2', a_sol_d_power, a_sol_r_power, 'P_e_SOL / q_P_SOL'),
    _comparison('model defaults', 'f_e', '', f_e, f_e, 'separatrix.py default'),
    _comparison('model defaults', 'f_out', '', f_out, f_out, 'separatrix.py default'),
    _comparison('model defaults', 'f_cond', '', f_cond, f_cond, 'separatrix.py default'),
    _comparison('model defaults', 'Zeff', '', zeff_d, zeff_r, 'general.csv / repo scalar'),
    _comparison('conduction model', 'k0', 'W/eV^3.5', k0_d, k0_r, '2600/(0.672 + 0.076*sqrt(Zeff) + 0.252*Zeff)'),
    _comparison('model defaults', 'q_distribution', '', np.nan, float(repo_case['q_distribution']), 'repo scalar'),
]

qsol_comparison = pd.DataFrame(comparison_rows)
qsol_comparison.to_csv('qsol_comparison.csv', index=False)
print('Direct scalar-state Davide versus repo comparison:')
display(qsol_comparison)


In [ ]:
davide_power['dWdt'] = davide_csv_dwdt * 1e6
print(davide_power['dWdt'])
result = compute_physics_separatrix(
        davide_x,
        davide_te,
        davide_Ne,
        power_balance=davide_power,
        geometry=shared_geometry,
        x_grid=davide_x_dense,
        fit_method=f'davide-csv-dwdt',
    )
result

In [ ]:
# Diagnose the remaining q_cond mismatch using the direct scalar model state.
repo_te_state = float(repo_case['Te'])
repo_te_sep = float(repo_case['Te_sep'])
repo_l_sol = float(repo_case['L_par'])
repo_k0 = float(repo_case['kappa_0'])
repo_q_distribution = float(repo_case['q_distribution'])
repo_q_cond_from_state = (2.0 / 7.0 * repo_q_distribution * repo_k0
                          * repo_te_state**3.5 / repo_l_sol) / 1e6
repo_q_cond_at_te_sep = (2.0 / 7.0 * repo_q_distribution * repo_k0
                         * repo_te_sep**3.5 / repo_l_sol) / 1e6
print(pd.DataFrame([
    {'quantity': 'Te used by repo q_par_SH', 'value': repo_te_state, 'unit': 'eV'},
    {'quantity': 'Te_sep reported by repo', 'value': repo_te_sep, 'unit': 'eV'},
    {'quantity': 'Te_sep in Davide CSV', 'value': te_sep_d, 'unit': 'eV'},
    {'quantity': 'q_cond stored in repo_case', 'value': q_cond_r, 'unit': 'MW/m^2'},
    {'quantity': 'q_cond reconstructed from repo_case[Te]', 'value': repo_q_cond_from_state, 'unit': 'MW/m^2'},
    {'quantity': 'q_cond reconstructed from repo_case[Te_sep]', 'value': repo_q_cond_at_te_sep, 'unit': 'MW/m^2'},
    {'quantity': 'q_cond Davide CSV', 'value': q_cond_d, 'unit': 'MW/m^2'},
]).to_string(index=False))
print({
    'q_cond_ratio_repo_over_davide': q_cond_r / q_cond_d,
    'temperature_ratio_to_3.5': (repo_te_state / te_sep_d) ** 3.5,
    'q_cond_ratio_if_using_repo_Te_sep': repo_q_cond_at_te_sep / q_cond_d,
})


In [ ]:
display(qsol_comparison.tail(30))

In [ ]:
shared_physics_inputs['geometry']

## Sensibility to my fit to the ten parameters

In [ ]:
# Diagnose the current mtanh fit by region and by nearby polynomial choices.
def summarize_residuals(label, data, fit, errors):
    residual = np.asarray(data, dtype=float) - np.asarray(fit, dtype=float)
    valid = np.isfinite(residual) & np.isfinite(errors) & (np.asarray(errors) > 0)
    return {
        'label': label,
        'points': int(valid.sum()),
        'bias_eV': float(np.mean(residual[valid])),
        'mae_eV': float(np.mean(np.abs(residual[valid]))),
        'rmse_eV': float(np.sqrt(np.mean(residual[valid] ** 2))),
        'weighted_rmse': float(np.sqrt(np.mean((residual[valid] / np.asarray(errors)[valid]) ** 2))),
    }

current_params = np.asarray(mtanh_te[0], dtype=float)
print('Parameter names and values:')
for name, value in zip(
    ['x0', 'w', 'a_tail', 'b_tail', 'c_tail', 'd_tail', 'a_top', 'b_top', 'c_top', 'd_top'],
    current_params,
):
    print(f'  {name:>7s} = {value:.8g}')

current_core_fit = modified_tanh(core['r_midplane'].to_numpy(float), *current_params)
current_edge_fit = modified_tanh(edge['r_midplane'].to_numpy(float), *current_params)
print('\nCurrent-fit residuals by region:')
print(summarize_residuals('current core', core['Te'], current_core_fit, core['Te_err']))
print(summarize_residuals('current edge', edge['Te'], current_edge_fit, edge['Te_err']))

fit_grid = np.linspace(current_r.min(), current_r.max(), 1000)
fit_grid_values = modified_tanh(fit_grid, *current_params)
print('\nPhysicality checks on the current radial domain:')
print({
    'w_positive': bool(current_params[1] > 0),
    'fit_min_eV': float(np.min(fit_grid_values)),
    'fit_max_eV': float(np.max(fit_grid_values)),
    'negative_fit_points': int(np.sum(fit_grid_values < 0)),
})

# Compare nearby model complexities using the same input data and uncertainties.
complexity_results = []
for deg_tail, deg_top in [(0, 1), (1, 1), (1, 2), (2, 2), (2, 3), (3, 3)]:
    params, _ = fit_modified_tanh_profile(
        x_te, y_te, yerr_te,
        deg_tail=deg_tail,
        deg_top=deg_top,
        fix_a_tail=None,
    )
    fitted_values = modified_tanh(x_te, *params)
    metrics = summarize_residuals(
        f'tail={deg_tail}, top={deg_top}', y_te, fitted_values, yerr_te
    )
    metrics['x0'] = float(params[0])
    metrics['w'] = float(params[1])
    complexity_results.append(metrics)

print('\nNearby degree-choice comparison on current data:')
print(pd.DataFrame(complexity_results).to_string(index=False))

## Analyse Davide's Errors

In [ ]:
# Compare Davide's raw TS errors, fit uncertainties, and current Te errors.
def error_summary(label, values):
    values = np.asarray(values, dtype=float)
    finite = values[np.isfinite(values)]
    return {
        'label': label,
        'points': int(finite.size),
        'min_eV': float(np.min(finite)),
        'median_eV': float(np.median(finite)),
        'mean_eV': float(np.mean(finite)),
        'max_eV': float(np.max(finite)),
    }

def compare_error_sources(reference_label, reference_profile, current_profile):
    reference_r = reference_profile['Rmaj [m]'].to_numpy(float)
    reference_te_err = 1e3 * reference_profile.iloc[:, 3].to_numpy(float)
    current_r = current_profile['r_midplane'].to_numpy(float)
    current_te_err = current_profile['Te_err'].to_numpy(float)

    nearest_index = np.abs(reference_r[:, None] - current_r[None, :]).argmin(axis=1)
    radial_delta = current_r[nearest_index] - reference_r
    matched = np.abs(radial_delta) <= 0.005
    matched_reference_err = reference_te_err[matched]
    matched_current_err = current_te_err[nearest_index[matched]]
    valid = (
        np.isfinite(matched_reference_err)
        & np.isfinite(matched_current_err)
        & (matched_reference_err > 0)
        & (matched_current_err > 0)
    )
    ratio = matched_reference_err[valid] / matched_current_err[valid]

    return {
        'region': reference_label,
        'Davide_points': int(reference_r.size),
        'current_points': int(current_r.size),
        'matched_within_5mm': int(np.sum(matched)),
        'Davide_dTe_over_current_Te_err_median': float(np.median(ratio)),
        'Davide_dTe_over_current_Te_err_min': float(np.min(ratio)),
        'Davide_dTe_over_current_Te_err_max': float(np.max(ratio)),
    }

print('Davide raw TS Te errors:')
for label in ['Te_coreTS', 'Te_edgeTS']:
    profile = reference_profiles[label]
    print(error_summary(
        f"Davide {label} dTe_e",
        1e3 * profile.iloc[:, 3].to_numpy(float),
    ))

print('\nDavide saved fit uncertainty:')
print(error_summary('Davide dfit_data', davide_te_err))

print('\nCurrent workflow Te errors:')
print(error_summary('Current core Te_err', core['Te_err']))
print(error_summary('Current edge Te_err', edge['Te_err']))

print('\nMatched raw-error ratios:')
error_comparisons = {
    'core': compare_error_sources(
        'core', reference_profiles['Te_coreTS'], core
    ),
    'edge': compare_error_sources(
        'edge', reference_profiles['Te_edgeTS'], edge
    ),
}
print(error_comparisons)

# Show the edge comparison sorted by radius, where the weighting question matters most.
davide_edge_error_table = reference_profiles['Te_edgeTS'][
    ['Rmaj [m]', 'Te_e [keV]', 'dTe_e [keV]']
].copy()
davide_edge_error_table['Davide_Te_eV'] = 1e3 * davide_edge_error_table['Te_e [keV]']
davide_edge_error_table['Davide_dTe_eV'] = 1e3 * davide_edge_error_table['dTe_e [keV]']
current_edge_error_table = edge[['r_midplane', 'Te', 'Te_err']].copy()
current_edge_error_table = current_edge_error_table.rename(
    columns={'r_midplane': 'current_R_m', 'Te': 'current_Te_eV', 'Te_err': 'current_Te_err_eV'}
)
edge_error_table = davide_edge_error_table.assign(
    current_index=np.abs(
        davide_edge_error_table['Rmaj [m]'].to_numpy()[:, None]
        - current_edge_error_table['current_R_m'].to_numpy()[None, :]
    ).argmin(axis=1)
)
edge_error_table['current_R_m'] = current_edge_error_table.iloc[
    edge_error_table['current_index'], 0
].to_numpy()
edge_error_table['current_Te_eV'] = current_edge_error_table.iloc[
    edge_error_table['current_index'], 1
].to_numpy()
edge_error_table['current_Te_err_eV'] = current_edge_error_table.iloc[
    edge_error_table['current_index'], 2
].to_numpy()
edge_error_table['dTe_ratio'] = (
    edge_error_table['Davide_dTe_eV'] / edge_error_table['current_Te_err_eV']
)
print('\nEdge error comparison by radius:')
print(edge_error_table[
    [
        'Rmaj [m]', 'Davide_Te_eV', 'Davide_dTe_eV',
        'current_R_m', 'current_Te_eV', 'current_Te_err_eV', 'dTe_ratio',
    ]
].sort_values('Rmaj [m]').to_string(index=False))

# Uncertainty Models

In [ ]:
# Compare three structured Te uncertainty sweeps.
# The raw Te_err column remains unchanged; only the fit weights vary.
sweep_families = {
    'fixed-only': fixed_error_sweep,
    'relative-only': relative_error_sweep,
    'combined': combined_error_sweep,
}
scenario_fits = {}
scenario_metrics = []
scenario_x = np.linspace(
    max(davide_x.min(), x_te.min()),
    min(davide_x.max(), x_te.max()),
    500,
)
scenario_reference_te = np.interp(scenario_x, davide_x, davide_te)
scenario_ne = modified_tanh(scenario_x, *np.asarray(mtanh_ne[0], dtype=float))

for family_name, family in sweep_families.items():
    for scenario_name, error_values in family.items():
        scenario_df = augment_measurement_errors(
            mapped,
            quantity='Te',
            error_column=te_error_config['source_column'],
            fixed_error=error_values['fixed_error'],
            relative_error=error_values['relative_error'],
            output_column='Te_err_scenario',
        )
        scenario_errors = scenario_df['Te_err_scenario'].to_numpy(float)
        scenario_fit = fit_modified_tanh_profile(
            x_te, y_te, scenario_errors,
            deg_tail=mtanh_config['Te']['deg_tail'],
            deg_top=mtanh_config['Te']['deg_top'],
            fix_a_tail=mtanh_config['Te']['fix_a_tail'],
        )
        scenario_curve = modified_tanh(
            scenario_x, *np.asarray(scenario_fit[0], dtype=float)
        )
        scenario_physics = compute_physics_separatrix(
            scenario_x,
            scenario_curve,
            scenario_ne,
            power_balance=physics_inputs['power_balance'],
            geometry=physics_inputs['geometry'],
            x_grid=scenario_x,
            fit_method='mtanh',
        )
        difference = scenario_curve - scenario_reference_te
        scenario_fits[(family_name, scenario_name)] = {
            'params': scenario_fit[0],
            'covariance': scenario_fit[1],
            'errors': scenario_errors,
            'curve': scenario_curve,
            'physics': scenario_physics,
        }
        scenario_metrics.append({
            'family': family_name,
            'scenario': scenario_name,
            'fixed_error_eV': error_values['fixed_error'],
            'relative_error': error_values['relative_error'],
            'Rsep_m': float(scenario_physics['Rsep']),
            'delta_Rsep_mm_vs_original': np.nan,
            'SOL_bias_vs_Davide_eV': float(np.mean(difference[scenario_x >= scenario_physics['Rsep']])),
            'SOL_MAE_vs_Davide_eV': float(np.mean(np.abs(difference[scenario_x >= scenario_physics['Rsep']]))),
        })

scenario_metrics = pd.DataFrame(scenario_metrics)
original_rsep = scenario_metrics.loc[
    scenario_metrics['scenario'].eq('fixed_0eV'), 'Rsep_m'
].iloc[0]
scenario_metrics['delta_Rsep_mm_vs_original'] = (
    1e3 * (scenario_metrics['Rsep_m'] - original_rsep)
)
print(scenario_metrics.to_string(index=False))

family_colors = {
    'fixed-only': plt.cm.Blues,
    'relative-only': plt.cm.Oranges,
    'combined': plt.cm.Greens,
}
fig_error_sweep, axes = plt.subplots(2, 2, figsize=(14, 9), sharex=False)
axes = axes.ravel()
for axis, (family_name, family) in zip(axes[:3], sweep_families.items()):
    colors = family_colors[family_name](np.linspace(0.45, 0.95, len(family)))
    axis.errorbar(
        current_r, current_te, yerr=current_te_err,
        fmt='.', color='0.55', alpha=0.25, label='TS data, raw errors',
    )
    axis.plot(davide_x, davide_te, 'k--', linewidth=2, label="Davide's mtanh")
    for color, scenario_name in zip(colors, family):
        scenario_data = scenario_fits[(family_name, scenario_name)]
        scenario_rsep = scenario_data['physics']['Rsep']
        axis.plot(
            scenario_x, scenario_data['curve'], color=color, linewidth=1.8,
            label=f"{scenario_name} (Rsep={scenario_rsep:.4f} m)",
        )
        axis.axvline(scenario_rsep, color=color, linestyle=':', alpha=0.45)
    axis.axvline(original_rsep, color='0.2', linestyle='-.', linewidth=1, label='original Rsep')
    axis.set_title(f'{family_name} error sweep')
    axis.set_xlabel('R_midplane [m]')
    axis.set_ylabel('Te [eV]')
    axis.set_xlim(2.11, 2.19)
    axis.set_ylim(0, 1100)
    axis.grid(True, alpha=0.25)
    axis.legend(loc='best', fontsize=7)

family_offsets = {'fixed-only': 0, 'relative-only': 7, 'combined': 13}
all_positions = []
all_labels = []
for family_name, family in sweep_families.items():
    family_metrics = scenario_metrics[scenario_metrics['family'] == family_name]
    x_positions = np.arange(len(family_metrics)) + family_offsets[family_name]
    all_positions.extend(x_positions)
    all_labels.extend(family_metrics['scenario'])
    axes[3].plot(
        x_positions, family_metrics['delta_Rsep_mm_vs_original'],
        'o-', label=family_name,
    )
axes[3].set_xticks(all_positions, all_labels)
axes[3].axhline(0.0, color='0.3', linewidth=1)
axes[3].set_title('Separatrix displacement')
axes[3].set_ylabel(r'$\Delta R_{sep}$ [mm] vs original')
axes[3].set_xlabel('Error scenario')
axes[3].tick_params(axis='x', rotation=45)
axes[3].grid(True, alpha=0.25)
axes[3].legend(loc='best')
fig_error_sweep.tight_layout()
plt.show()

In [ ]:
# Repeat the Te uncertainty sweep for the B-spline fit.
# Only Te_err is changed between scenarios; the Ne data and errors stay fixed.
spline_scenario_fits = {}
spline_scenario_metrics = []
spline_reference_x = np.linspace(
    max(davide_x.min(), ds_out['R_mid_spl'].min()),
    min(davide_x.max(), ds_out['R_mid_spl'].max()),
    500,
)
spline_reference_te = np.interp(spline_reference_x, davide_x, davide_te)

for family_name, family in sweep_families.items():
    for scenario_name, error_values in family.items():
        scenario_df = augment_measurement_errors(
            mapped,
            quantity='Te',
            error_column=te_error_config['source_column'],
            fixed_error=error_values['fixed_error'],
            relative_error=error_values['relative_error'],
            output_column='Te_err_scenario',
        )
        scenario_df['Te_err'] = scenario_df['Te_err_scenario']
        scenario_df = scenario_df.drop(columns=['Te_err_scenario'])
        scenario_spline_input = (
            scenario_df.rename(columns={'r_midplane': 'R_mid', 'Ne': 'ne', 'Ne_err': 'ne_err'})
            .dropna(subset=['R_mid', 'ne', 'ne_err', 'Te', 'Te_err'])
        )
        scenario_ds_out, scenario_knots, scenario_p_ne, scenario_pcov_ne, scenario_p_te, scenario_pcov_te = fit_mbsplt_ne_Te_TS(
            scenario_spline_input,
            dR_fine=spline_config['dR_fine'],
            sigma_conv=spline_config['sigma_conv'],
            R_knots=knots,
            dR_channel_max=spline_config['dR_channel_max'],
            max_step_scale=spline_config['max_step_scale'],
            use_errors=spline_config['use_errors'],
        )
        scenario_x_spline = scenario_ds_out['R_mid_spl'].to_numpy(float)
        scenario_curve = scenario_ds_out['Te_spl'].to_numpy(float)
        scenario_ne_curve = scenario_ds_out['ne_spl'].to_numpy(float)
        scenario_physics = compute_physics_separatrix(
            scenario_x_spline,
            scenario_curve,
            scenario_ne_curve,
            power_balance=physics_inputs['power_balance'],
            geometry=physics_inputs['geometry'],
            x_grid=scenario_x_spline,
            fit_method='spline',
        )
        common = (
            (spline_reference_x >= scenario_physics['Rsep'])
            & np.isfinite(spline_reference_te)
        )
        scenario_interp = np.interp(spline_reference_x, scenario_x_spline, scenario_curve)
        difference = scenario_interp - spline_reference_te
        spline_scenario_fits[(family_name, scenario_name)] = {
            'ds_out': scenario_ds_out,
            'params_ne': scenario_p_ne,
            'params_te': scenario_p_te,
            'physics': scenario_physics,
        }
        spline_scenario_metrics.append({
            'family': family_name,
            'scenario': scenario_name,
            'fixed_error_eV': error_values['fixed_error'],
            'relative_error': error_values['relative_error'],
            'Rsep_m': float(scenario_physics['Rsep']),
            'SOL_bias_vs_Davide_eV': float(np.mean(difference[common])),
            'SOL_MAE_vs_Davide_eV': float(np.mean(np.abs(difference[common]))),
        })

spline_scenario_metrics = pd.DataFrame(spline_scenario_metrics)
spline_original_rsep = spline_scenario_metrics.loc[
    spline_scenario_metrics['scenario'].eq('fixed_0eV'), 'Rsep_m'
].iloc[0]
spline_scenario_metrics['delta_Rsep_mm_vs_original'] = (
    1e3 * (spline_scenario_metrics['Rsep_m'] - spline_original_rsep)
)
print('Spline Te uncertainty sweep:')
print(spline_scenario_metrics.to_string(index=False))

fig_spline_error_sweep, spline_axes = plt.subplots(2, 2, figsize=(14, 9), sharex=False)
spline_axes = spline_axes.ravel()
for axis, (family_name, family) in zip(spline_axes[:3], sweep_families.items()):
    colors = family_colors[family_name](np.linspace(0.45, 0.95, len(family)))
    axis.errorbar(
        current_r, current_te, yerr=current_te_err,
        fmt='.', color='0.55', alpha=0.25, label='TS data, raw errors',
    )
    axis.plot(davide_x, davide_te, 'k--', linewidth=2, label="Davide's mtanh")
    for color, scenario_name in zip(colors, family):
        scenario_data = spline_scenario_fits[(family_name, scenario_name)]
        scenario_profile = scenario_data['ds_out']
        scenario_rsep = scenario_data['physics']['Rsep']
        axis.plot(
            scenario_profile['R_mid_spl'], scenario_profile['Te_spl'],
            color=color, linewidth=1.8,
            label=f"{scenario_name} (Rsep={scenario_rsep:.4f} m)",
        )
        axis.axvline(scenario_rsep, color=color, linestyle=':', alpha=0.45)
    axis.axvline(spline_original_rsep, color='0.2', linestyle='-.', linewidth=1, label='original spline Rsep')
    axis.set_title(f'{family_name} spline error sweep')
    axis.set_xlabel('R_midplane [m]')
    axis.set_ylabel('Te [eV]')
    axis.set_xlim(2.11, 2.19)
    axis.set_ylim(0, 1100)
    axis.grid(True, alpha=0.25)
    axis.legend(loc='best', fontsize=7)

spline_family_offsets = {'fixed-only': 0, 'relative-only': 7, 'combined': 13}
spline_positions = []
spline_labels = []
for family_name, family in sweep_families.items():
    family_metrics = spline_scenario_metrics[spline_scenario_metrics['family'] == family_name]
    x_positions = np.arange(len(family_metrics)) + spline_family_offsets[family_name]
    spline_positions.extend(x_positions)
    spline_labels.extend(family_metrics['scenario'])
    spline_axes[3].plot(
        x_positions, family_metrics['delta_Rsep_mm_vs_original'],
        'o-', label=family_name,
    )
spline_axes[3].set_xticks(spline_positions, spline_labels)
spline_axes[3].axhline(0.0, color='0.3', linewidth=1)
spline_axes[3].set_title('Spline separatrix displacement')
spline_axes[3].set_ylabel(r'$\Delta R_{sep}$ [mm] vs original spline')
spline_axes[3].set_xlabel('Error scenario')
spline_axes[3].tick_params(axis='x', rotation=45)
spline_axes[3].grid(True, alpha=0.25)
spline_axes[3].legend(loc='best')
fig_spline_error_sweep.tight_layout()
plt.show()

In [ ]:
# Three-panel comparison:
#   upper: lambda_Te
#   middle: qSOL and qcond
#   lower: existing Te comparison plot

davide_physics = 

import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

fig_three_panel, axes = plt.subplots(
    3,
    1,
    figsize=(10, 14),
    sharex=True,
    gridspec_kw={'height_ratios': [1, 1, 1.4]},
)

ax_lambda, ax_q, ax_te = axes

# ------------------------------------------------------------------
# Upper panel: lambda_Te
# ------------------------------------------------------------------
ax_lambda.plot(
    spline_result['R_grid'],
    1e3 * spline_result['lam_Te_grid'],
    color='green',
    linewidth=1.8,
    label='Spline',
)

ax_lambda.plot(
    mtanh_result['R_grid'],
    1e3 * mtanh_result['lam_Te_grid'],
    color='red',
    linewidth=1.8,
    linestyle='--',
    label='Project mtanh',
)

ax_lambda.plot(
    davide_physics['R_grid'],
    1e3 * davide_physics['lam_Te_grid'],
    color='black',
    linewidth=1.8,
    linestyle=':',
    label="Davide's mtanh",
)

ax_lambda.set_ylabel(r'$\lambda_{Te}$ [mm]')
ax_lambda.set_title('Electron-temperature scale length')
ax_lambda.set_xlim(2.10, 2.19)
ax_lambda.set_ylim(0,20)
ax_lambda.grid(True, alpha=0.25)
ax_lambda.legend(loc='best')

# ------------------------------------------------------------------
# Middle panel: qSOL and qcond
# ------------------------------------------------------------------
ax_q.plot(
    spline_result['R_grid'],
    spline_result['q_par_P_SOL_grid'] / 1e9,
    color='green',
    linewidth=1.8,
    label='Spline qSOL',
)

ax_q.plot(
    spline_result['R_grid'],
    spline_result['q_par_SH_grid'] / 1e9,
    color='green',
    linewidth=1.8,
    linestyle='--',
    label='Spline qcond',
)

ax_q.plot(
    mtanh_result['R_grid'],
    mtanh_result['q_par_P_SOL_grid'] / 1e9,
    color='red',
    linewidth=1.8,
    label='Project mtanh qSOL',
)

ax_q.plot(
    mtanh_result['R_grid'],
    mtanh_result['q_par_SH_grid'] / 1e9,
    color='red',
    linewidth=1.8,
    linestyle='--',
    label='Project mtanh qcond',
)

ax_q.plot(
    davide_physics['R_grid'],
    davide_physics['q_par_P_SOL_grid'] / 1e9,
    color='black',
    linewidth=1.8,
    linestyle=':',
    label="Davide's mtanh qSOL",
)

ax_q.set_ylabel(r'$q$ [GW/m$^2$]')
ax_q.set_title('Parallel heat-flux estimates')
ax_q.set_xlim(2.10, 2.19)
ax_q.set_ylim(0,1.5)
ax_q.grid(True, alpha=0.25)
ax_q.legend(loc='best', fontsize=8)

# ------------------------------------------------------------------
# Lower panel: existing Te comparison plot
# ------------------------------------------------------------------
# The existing comparison is created first, then transferred into
# the lower panel of the combined figure.

fig_compare, ax_compare = plotting_module.plot_profile_comparison_zoomed(
    2.10,
    2.19,
    core['r_midplane'],
    edge['r_midplane'],
    core['Te'],
    edge['Te'],
    core['Te_err'],
    edge['Te_err'],
    ds_out,
    'Te',
    mtanh_popt=mtanh_te[0],
    knot_pos=knots,
    Rsep_eqh=eqh_result['Rsep_eqh'],
    Rsep_spline=spline_result['Rsep'],
    Rsep_mtanh=mtanh_result['Rsep'],
    nshot=shot_id,
    interval=interval,
    directory=str(output_directory),
    fig_name=make_fig_name(
        shot_id,
        equilibrium_name,
        len(knots),
        interval,
        'te_project_vs_davide',
    ),
    y_labels='Te [eV]',
    v_factor=1,
    h_factor=1.5,
    save_fig=False,
    show=False,
    close=False,
)

from matplotlib.lines import Line2D

davide_curve_line, = ax_compare.plot(
    davide_x,
    davide_te,
    color='black',
    linestyle='--',
    linewidth=1.8,
    label="Davide mtanh",
    zorder=6,
)

davide_rsep_line = ax_compare.axvline(
    davide_rsep,
    color='black',
    linestyle=':',
    linewidth=1.2,
    label=f'Davide Rsep = {davide_rsep:.4f} m',
    zorder=4,
)

davide_rsep_line = ax_compare.axvline(
    2.143,
    color='magenta',
    linestyle=':',
    linewidth=1.2,
    label='Davide Rsep (paper) = 2.143 m',
    zorder=4,
)

legend_handles = [
    Line2D(
        [0],
        [0],
        color='green',
        linewidth=1.8,
        label='Spline',
    ),
    Line2D(
        [0],
        [0],
        color='red',
        linewidth=1.8,
        label='My mtanh',
    ),
    Line2D(
        [0],
        [0],
        color='black',
        linestyle='--',
        linewidth=1.8,
        label='Davide mtanh',
    ),
    Line2D(
        [0],
        [0],
        color='green',
        linestyle=':',
        linewidth=1.8,
        label='Rsep spline',
    ),
    Line2D(
        [0],
        [0],
        color='red',
        linestyle=':',
        linewidth=1.8,
        label='Rsep My mtanh',
    ),
    Line2D(
        [0],
        [0],
        color='blue',
        linestyle=':',
        linewidth=1.8,
        label='Rsep EQH',
    ),
    Line2D(
        [0],
        [0],
        color='black',
        linestyle=':',
        linewidth=1.8,
        label='Rsep Davide (calculated)',
    ),
    Line2D(
        [0],
        [0],
        color='magenta',
        linestyle=':',
        linewidth=1.8,
        label='Rsep Davide (paper)',
    ),
]

ax_compare.legend(handles=legend_handles, loc='upper right')

print({
    'Rsep_eqh_m': float(eqh_result['Rsep_eqh']),
    'Rsep_spline_m': float(spline_result['Rsep']),
    'Rsep_project_mtanh_m': float(mtanh_result['Rsep']),
    'Rsep_davide_mtanh_m': davide_rsep,
})

# Copy the rendered artists from the existing Te plot to the lower axis.
for artist in ax_compare.get_children():
    if hasattr(artist, 'get_xdata') and hasattr(artist, 'get_ydata'):
        try:
            ax_te.plot(
                artist.get_xdata(),
                artist.get_ydata(),
                color=artist.get_color(),
                linestyle=artist.get_linestyle(),
                linewidth=artist.get_linewidth(),
                marker=artist.get_marker(),
                label=artist.get_label(),
            )
        except (AttributeError, TypeError):
            pass

ax_te.set_xlim(2.10, 2.19)
ax_te.set_ylim(-300,1500)
ax_te.set_xlabel('R_midplane [m]')
ax_te.set_ylabel('Te [eV]')
ax_te.set_title('Electron-temperature profile comparison')
ax_te.grid(True, alpha=0.25)

# Add the same main Te curves explicitly to guarantee they appear.
ax_te.plot(
    ds_out['R_mid_spl'],
    ds_out['Te_spl'],
    color='green',
    linewidth=1.8,
    label='Spline',
)

ax_te.plot(
    R_mtanh,
    modified_tanh(R_mtanh, *np.asarray(mtanh_te[0], dtype=float)),
    color='red',
    linewidth=1.8,
    label='My mtanh',
)

ax_te.plot(
    davide_x,
    davide_te,
    color='black',
    linestyle='--',
    linewidth=1.8,
    label='Davide mtanh',
)

for radius, color in [
    (float(eqh_result['Rsep_eqh']), 'blue'),
    (float(spline_result['Rsep']), 'green'),
    (float(mtanh_result['Rsep']), 'red'),
    (davide_rsep, 'black'),
    (2.143, 'magenta'),
]:
    ax_te.axvline(
        radius,
        color=color,
        linestyle=':',
        linewidth=1.1,
        alpha=0.8,
    )

ax_te.legend(handles=legend_handles, loc='upper right', fontsize=8)

# Close the temporary figure created by plot_profile_comparison_zoomed.
plt.close(fig_compare)

fig_three_panel.tight_layout()

if save_figures:
    fig_three_panel.savefig(
        output_directory / make_fig_name(
            shot_id,
            equilibrium_name,
            len(knots),
            interval,
            'lambda_q_te_three_panel',
        ),
        bbox_inches='tight',
    )

if show_figures:
    plt.show()
else:
    plt.close(fig_three_panel)

## qSOL derivative-method comparison

The paper evaluates lambda_Te from the fitted mtanh profile. The first result below reproduces the current exported-curve path, which differentiates the 550 sampled fit values with `np.gradient`; the second uses a smooth cubic derivative of the same exported curve as an analytic-derivative proxy.

In [ ]:
from scipy.interpolate import CubicSpline

def summarize_qsol_derivative(label, result):
    q_values = np.asarray(result['q_par_P_SOL_grid'], dtype=float)
    valid = np.isfinite(q_values)
    index = np.flatnonzero(valid)[np.argmax(q_values[valid])]
    return {
        'method': label,
        'Rsep_m': float(result['Rsep']),
        'qSOL_max_GW_m2': float(q_values[index] / 1e9),
        'R_at_qSOL_max_m': float(result['R_grid'][index]),
        'lambda_Te_at_qSOL_max_mm': float(result['lam_Te_grid'][index] * 1e3),
    }

# Current path: make_profile computes dTe/dR with np.gradient on the exported samples.
davide_gradient = compute_physics_separatrix(
    davide_x, davide_te, Ne=None,
    power_balance=physics_inputs['power_balance'],
    geometry=physics_inputs['geometry'],
    x_grid=davide_x_dense, fit_method='davide-gradient',
)

# Smooth derivative path: differentiate a cubic interpolant of the same fit curve.
davide_curve_spline = CubicSpline(davide_x, davide_te)
davide_analytic_proxy = compute_physics_separatrix(
    davide_x, davide_te, Ne=None,
    power_balance=physics_inputs['power_balance'],
    geometry=physics_inputs['geometry'],
    dTe_dR=davide_curve_spline(davide_x, 1),
    x_grid=davide_x_dense, fit_method='davide-cubic-derivative',
)

comparison = pd.DataFrame([
    summarize_qsol_derivative('exported curve + np.gradient', davide_gradient),
    summarize_qsol_derivative('same curve + smooth derivative', davide_analytic_proxy),
]) 
print(comparison.to_string(index=False))

def values_at(result, radius):
    q_value = np.interp(radius, result['R_grid'], result['q_par_P_SOL_grid']) / 1e9
    lambda_value = np.interp(radius, result['R_grid'], result['lam_Te_grid']) * 1e3
    return {'R_m': radius, 'lambda_Te_mm': float(lambda_value), 'qSOL_GW_m2': float(q_value)}

print('At the paper separatrix (R=2.143 m):')
print(pd.DataFrame([
    values_at(davide_gradient, 2.143),
    values_at(davide_analytic_proxy, 2.143),
]).assign(method=['np.gradient', 'smooth derivative']).to_string(index=False))

gradient_peak = float(np.nanmax(davide_gradient['q_par_P_SOL_grid']))
smooth_peak = float(np.nanmax(davide_analytic_proxy['q_par_P_SOL_grid']))
print({
    'smooth_over_gradient_qSOL_peak': smooth_peak / gradient_peak,
    'gradient_over_smooth_qSOL_peak': gradient_peak / smooth_peak,
    'power_balance_P_SOL_MW': float(davide_gradient['P_SOL'] / 1e6),
    'electron_outer_conducted_power_MW': float(davide_gradient['P_e_SOL'] / 1e6),
    'power_fraction_product': 0.8 * 0.65 * 0.8,
})

# Knot-count and allocation sweep.

In [ ]:
# Compare spline shape and physics separatrix for balanced and edge-dominant knot allocations.
# Sweep total knot count around the current edge-dominant baseline, without changing Te errors.
knot_sweep_shifts = [-2, 0, 2, 4]
knot_sweep_strategies = ['balanced_quantile', 'edge_biased_quantile']
knot_sweep_results = []
knot_sweep_profiles = {}

for strategy in knot_sweep_strategies:
    for shift in knot_sweep_shifts:
        sweep_knots = compute_channel_knots(
            spline_input,
            preprocess_metadata['channels_before'],
            preprocess_metadata['channels_removed'],
            knot_count_shift=shift,
            strategy=strategy,
        )
        sweep_ds_out, sweep_knots, sweep_p_ne, sweep_pcov_ne, sweep_p_te, sweep_pcov_te = fit_mbsplt_ne_Te_TS(
            spline_input,
            #dR_fine=spline_config['dR_fine'],
            sigma_conv=spline_config['sigma_conv'],
            R_knots=sweep_knots,
            #dR_channel_max=spline_config['dR_channel_max'],
            #max_step_scale=spline_config['max_step_scale'],
            use_errors=spline_config['use_errors'],
        )
        sweep_r = sweep_ds_out['R_mid_spl'].to_numpy(float)
        sweep_te = sweep_ds_out['Te_spl'].to_numpy(float)
        sweep_ne = sweep_ds_out['ne_spl'].to_numpy(float)
        sweep_physics = compute_physics_separatrix(
            sweep_r,
            sweep_te,
            sweep_ne,
            power_balance=physics_inputs['power_balance'],
            geometry=physics_inputs['geometry'],
            x_grid=sweep_r,
            fit_method='spline',
        )
        key = (strategy, shift)
        knot_sweep_profiles[key] = {
            'ds_out': sweep_ds_out,
            'knots': sweep_knots,
            'physics': sweep_physics,
        }
        knot_sweep_results.append({
            'strategy': strategy,
            'knot_shift': shift,
            'n_knots': len(sweep_knots),
            'Rsep_m': float(sweep_physics['Rsep']),
        })

knot_sweep_results = pd.DataFrame(knot_sweep_results)
baseline_rsep = float(knot_sweep_results.loc[
    (knot_sweep_results['strategy'] == 'edge_biased_quantile')
    & (knot_sweep_results['knot_shift'] == 2),
    'Rsep_m',
].iloc[0])
knot_sweep_results['delta_Rsep_mm_vs_edge12'] = (
    1e3 * (knot_sweep_results['Rsep_m'] - baseline_rsep)
)
print('Spline knot-count/allocation sweep:')
print(knot_sweep_results.to_string(index=False))

fig_knot_sweep, knot_axes = plt.subplots(2, 2, figsize=(14, 10), sharex=False)
knot_axes = knot_axes.ravel()
colors = {'balanced_quantile': 'tab:blue', 'edge_biased_quantile': 'tab:orange'}
styles = {-2: ':', 0: '--', 2: '-', 4: '-.'}
strategy_titles = {
    'balanced_quantile': 'Balanced quantile knots',
    'edge_biased_quantile': 'Edge-dominant quantile knots',
}

for axis_index, strategy in enumerate(knot_sweep_strategies):
    axis = knot_axes[axis_index]
    for shift in knot_sweep_shifts:
        profile = knot_sweep_profiles[(strategy, shift)]
        sweep_profile = profile['ds_out']
        label = f'shift={shift:+d}, n={len(profile["knots"])}'
        axis.plot(
            sweep_profile['R_mid_spl'],
            sweep_profile['Te_spl'],
            color=colors[strategy],
            linestyle=styles[shift],
            linewidth=1.8,
            label=label,
        )
        axis.axvline(
            profile['physics']['Rsep'],
            color=colors[strategy],
            linestyle=styles[shift],
            alpha=0.35,
        )
    axis.plot(davide_x, davide_te, color='black', linestyle='--', linewidth=2, label="Davide's mtanh")
    axis.errorbar(
        current_r,
        current_te,
        yerr=current_te_err,
        fmt='.',
        color='0.45',
        alpha=0.25,
        label='TS data',
    )
    axis.set(
        xlabel='R_midplane [m]',
        ylabel='Te [eV]',
        title=f'{strategy_titles[strategy]}: curves and Rsep',
        xlim=(2.06, 2.19),
        ylim=(0, 1200),
    )
    axis.grid(True, alpha=0.25)
    axis.legend(fontsize=8, loc='best')

for strategy in knot_sweep_strategies:
    subset = knot_sweep_results[knot_sweep_results['strategy'] == strategy]
    knot_axes[2].plot(
        subset['n_knots'],
        subset['Rsep_m'],
        'o-',
        color=colors[strategy],
        label=strategy.replace('_quantile', ''),
    )
knot_axes[2].axhline(baseline_rsep, color='0.25', linestyle=':', label='edge-dominant 12-knot baseline')
knot_axes[2].set(
    xlabel='Number of knots',
    ylabel='Rsep [m]',
    title='Separatrix position',
)
knot_axes[2].grid(True, alpha=0.25)
knot_axes[2].legend(loc='best')

for strategy in knot_sweep_strategies:
    subset = knot_sweep_results[knot_sweep_results['strategy'] == strategy]
    knot_axes[3].plot(
        subset['n_knots'],
        subset['delta_Rsep_mm_vs_edge12'],
        'o-',
        color=colors[strategy],
        label=strategy.replace('_quantile', ''),
    )
knot_axes[3].axhline(0.0, color='0.25', linewidth=1)
knot_axes[3].set(
    xlabel='Number of knots',
    ylabel='Delta Rsep [mm]',
    title='Separatrix advance/decrease vs edge baseline',
)
knot_axes[3].grid(True, alpha=0.25)
knot_axes[3].legend(loc='best')
fig_knot_sweep.tight_layout()
plt.show()

## Export fit input
Export the filtered measurements used by the fits, together with a small JSON record of the analysis settings. The CSV is built from the same finite rows used to construct `spline_input` and retains the raw measurement errors.

In [ ]:
from datetime import datetime, timezone
import json

export_directory = project_root / 'exports'
export_directory.mkdir(parents=True, exist_ok=True)

# Keep exactly the finite rows passed to the spline, while retaining provenance and raw errors.
export_columns = [
    'system', 'channel_index', 'time', 'r_channel', 'z_channel',
    'rho_pol', 'r_midplane', 'Te', 'Te_err', 'Ne', 'Ne_err',
    'filter_reason',
]
available_columns = [column for column in export_columns if column in mapped.columns]
spline_rows = spline_input.index
export_data = mapped.loc[spline_rows, available_columns].copy()
export_data = export_data.sort_values(['time', 'r_midplane', 'system', 'channel_index'])
export_data.to_csv(export_directory / f'shot{shot_id}_fit_input.csv', index=False)

# Export Davide measurements with the same canonical names as the fit-input CSV.
davide_measurements = []
for system, profile_key, value_column, error_column in [
    ('core', 'Te_coreTS', 'Te_c [keV]', 'dTe_c [keV]'),
    ('edge', 'Te_edgeTS', 'Te_e [keV]', 'dTe_e [keV]'),
 ]:
    profile = reference_profiles[profile_key]
    davide_measurements.append(pd.DataFrame({
        'system': system,
        'r_midplane': profile['Rmaj [m]'].to_numpy(float),
        'Te': 1e3 * profile[value_column].to_numpy(float),
        'Te_err': 1e3 * profile[error_column].to_numpy(float),
    }))
davide_data = pd.concat(davide_measurements, ignore_index=True)
davide_data = davide_data.sort_values(['r_midplane', 'system']).reset_index(drop=True)
davide_data.to_csv(
    export_directory / f'shot{shot_id}_davide_measurements.csv',
    index=False,
 )

# Export Davide's saved Te mtanh curve separately from the measured points.
davide_fit_export = reference_profiles['Te_fit-mtanh'].copy()
davide_fit_export['R_m'] = davide_fit_export['Rmaj [m]'].astype(float)
davide_fit_export['Te_fit_eV'] = 1e3 * davide_fit_export['fit_data [keV]'].astype(float)
davide_fit_export['Te_fit_err_eV'] = 1e3 * davide_fit_export['dfit_data [keV]'].astype(float)
davide_fit_export.to_csv(
    export_directory / f'shot{shot_id}_davide_saved_mtanh.csv',
    index=False,
 )

spline_knots_export = pd.DataFrame({
    'knot_index': np.arange(len(knots), dtype=int),
    'R_knot_m': np.asarray(knots, dtype=float),
})
spline_knots_export.to_csv(
    export_directory / f'shot{shot_id}_spline_knots.csv',
    index=False,
)

export_metadata = {
    'shot_id': int(shot_id),
    'interval_s': {
        'begin': float(interval['t_begin']),
        'end': float(interval['t_end']),
    },
    'equilibrium': equilibrium_name,
    'diagnostic': diagnostic,
    'mapping_method': mapping_method,
    'r_edge_offset_m': float(r_edge_offset),
    'elm_filter': {
        'enabled': bool(use_elm_filter),
        'start_fraction': float(elm_cycle_start_fraction),
        'end_fraction': float(elm_cycle_end_fraction),
    },
    'rhop_filter': rhop_filter,
    'outlier_config': outlier_config,
    'te_error_config': {
        'enabled': te_error_config['enabled'],
        'fit_error_column': te_error_config['fit_error_column'],
        'exported_error_column': 'Te_err',
    },
    'spline_config': spline_config,
    'units': {
        'r_midplane': 'm',
        'Te': 'eV',
        'Te_err': 'eV',
        'Ne': 'm^-3',
        'Ne_err': 'm^-3',
    },
    'files': {
        'fit_input': f'shot{shot_id}_fit_input.csv',
        'davide_measurements': f'shot{shot_id}_davide_measurements.csv',
        'davide_saved_mtanh': f'shot{shot_id}_davide_saved_mtanh.csv',
        'spline_knots': f'shot{shot_id}_spline_knots.csv',
    },
    'rows': {
        'filtered_mapped': int(len(mapped)),
        'spline_input': int(len(spline_input)),
        'exported': int(len(export_data)),
        'core_exported': int((export_data['system'] == 'core').sum()),
        'edge_exported': int((export_data['system'] == 'edge').sum()),
        'davide_core_measurements': int(len(reference_profiles['Te_coreTS'])),
        'davide_edge_measurements': int(len(reference_profiles['Te_edgeTS'])),
        'davide_saved_mtanh': int(len(davide_fit_export)),
    },
    'created_utc': datetime.now(timezone.utc).isoformat(),
}
with (export_directory / f'shot{shot_id}_fit_input.json').open('w', encoding='utf-8') as handle:
    json.dump(export_metadata, handle, indent=2, default=str)

print(f"Exported {len(export_data)} rows to {export_directory / f'shot{shot_id}_fit_input.csv'}")
print(f"Exported {len(davide_data)} Davide measurement rows to {export_directory / f'shot{shot_id}_davide_measurements.csv'}")
print(f"Exported {len(davide_fit_export)} Davide saved-fit rows to {export_directory / f'shot{shot_id}_davide_saved_mtanh.csv'}")
print(f"Exported {len(spline_knots_export)} spline knots to {export_directory / f'shot{shot_id}_spline_knots.csv'}")
print(f"Wrote metadata to {export_directory / f'shot{shot_id}_fit_input.json'}")